In [1]:
import matplotlib.pyplot as plt
import numpy as np
import pickle

from scripts.simulation.sequence import simulate_sequences, downsample_sequences
import scripts.simulation.rank_correlation as rs 
from scripts.simulation.correlation_mean import add_within_clust_score
nrm = np.load("scripts/simulation/nrm.npy")

In [2]:
# PARAMETERS
params = {
    "n_neurons": 100000,
    "n_motifs": 20,
    "n_bins": 100,
    "n_sequences": 100,
    "sigma_range": (1,1),    # increasing range makes sequences less similar
    "vol_param": (0.1, 0.3),     # beta distribution:  0.07, 0.9
                                  # a inactive (lower -> more neurons inactive, if high longer seqs, mean len shifts)
                                  # b active (lower -> more neurons active, if <1, skewed to 1, if lower longer seqs, mean len shifts)
    "corr_mu": False,
    "rho_mu": 0.0,
    "corr_sigma": False,
    "rho_sig": 0.0,
    "corr_volume": False,
    "rho_vol": 0.0,
    "shuffle_order": False,       # shuffle order of sequences

}

In [3]:
#import cProfile
#import pstats
#import io
# Profile it
#pr = cProfile.Profile()
#pr.enable()
seqs, seqs_labels, spk_times, sequences, true_templates, mu, sigma, volume, densities, cdfs = simulate_sequences(**params, random_state=1, plot=False, batch_size=5000)
#pr.disable()

# Print results
# s = io.StringIO()
# ps = pstats.Stats(pr, stream=s).sort_stats('cumulative')
# ps.print_stats(20)  # Top 20 functions
# print(s.getvalue())

Processed batch: neurons 0-5000/100000
Processed batch: neurons 5000-10000/100000
Processed batch: neurons 10000-15000/100000
Processed batch: neurons 15000-20000/100000
Processed batch: neurons 20000-25000/100000
Processed batch: neurons 25000-30000/100000
Processed batch: neurons 30000-35000/100000
Processed batch: neurons 35000-40000/100000
Processed batch: neurons 40000-45000/100000
Processed batch: neurons 45000-50000/100000
Processed batch: neurons 50000-55000/100000
Processed batch: neurons 55000-60000/100000
Processed batch: neurons 60000-65000/100000
Processed batch: neurons 65000-70000/100000
Processed batch: neurons 70000-75000/100000
Processed batch: neurons 75000-80000/100000
Processed batch: neurons 80000-85000/100000
Processed batch: neurons 85000-90000/100000
Processed batch: neurons 90000-95000/100000
Processed batch: neurons 95000-100000/100000
Generated sequences for motif 0/20
Generated sequences for motif 1/20
Generated sequences for motif 2/20
Generated sequences 

In [ ]:
%reload_ext autoreload

In [6]:
with open(f'simulation_N{params["n_neurons"]}_sig_{params["sigma_range"]}_vol_{params["vol_param"]}_rhosig_{params["rho_sig"]}_rhovol_{params["rho_vol"]}.pkl', 'wb') as f:
    pickle.dump({
        "sequences": sequences,
        "volume": volume,
        'seqs': seqs,
        'seqs_labels': seqs_labels,
        'spike_times': spk_times,
    }, f)

In [4]:
# Test with min_length filter
seqs_downsampled_filtered, spk_times_downsampled_filt, volume_downsampled_filt = downsample_sequences(
    sequences, spk_times, volume, params["n_neurons"], n_neurons_keep=100, min_length=5, random_state=1
)

In [5]:
# rebuild labels for downsampled sequences using original motif labels from simulation
original_label_set = set(seqs_labels)
seqs_labels_downsampled_filt = [
    motif_id
    for motif_id, seq_list in seqs_downsampled_filtered.items()
    for _ in seq_list
]

In [6]:
with open(f'downsample_R100_N{params["n_neurons"]}_sig_{params["sigma_range"]}_vol_{params["vol_param"]}_rhosig_{params["rho_sig"]}_rhovol_{params["rho_vol"]}.pkl', 'wb') as f:
    pickle.dump({
        "seqs": seqs_downsampled_filtered,
        "volume": volume_downsampled_filt,
        'spike_times': spk_times_downsampled_filt,
        'seqs_labels': seqs_labels_downsampled_filt
    }, f)

In [ ]:
with open(f'simulation_N100000_sig_(0.02, 0.4)_vol_(0.07, 0.9)_rhosig_0.0_rhovol_0.0.pkl', 'rb') as f:
    data = pickle.load(f)

sequences = data["sequences"]
volume = data["volume"] 

with open(f'downsample_R100_N100000_sig_(0.02, 0.4)_vol_(0.07, 0.9)_rhosig_0.0_rhovol_0.0.pkl', 'rb') as f:
    data = pickle.load(f)
    
seqs_downsampled_filtered = data["seqs_down"]
volume_downsampled_filt = data["volume_down"] 

In [ ]:
# Filter motifs with more than 20 sequences
motif_ids_filtered = [m for m in seqs_downsampled_filtered.keys() if len(seqs_downsampled_filtered[m]) > 20]

seqs_downsampled_filtered = {m: seqs_downsampled_filtered[m] for m in motif_ids_filtered }
#spk_times_downsampled_filt = {m: spk_times_downsampled_filt[m] for m in motif_ids_filtered}
volume_downsampled_filt = {m: volume_downsampled_filt[m] for m in motif_ids_filtered}

print(f"Motifs after filtering (>20 sequences): {len(motif_ids_filtered)} out of {len(sequences)}")
print(f"Motifs kept: {sorted(motif_ids_filtered)}")

### Sigma - Neuron Orderding ###

In [ ]:
# Focus on one neuron and its firing order across sequences
# Compute the relative order of the neuron in each sequence and compare across sequences (0 = first, 1 = last)
neuron_id = seqs_downsampled_filtered[motif_ids_filtered[0]][0][0]  # Get a neuron ID from the first sequence of the first motif
active_seqs = []
seq_lengths = []
relative_orders = []  
neuron_indices = []   
#motif_ids = [m for m in seqs_downsampled.keys() if len(seqs_downsampled[m]) > 20]
for motif_id, seq_list in seqs_downsampled_filtered.items():
    for seq in seq_list:
        if neuron_id in seq:
            active_seqs.append(seq)
            seq_lengths.append(len(seq))
            # Find exact index using list or array indexing
            if isinstance(seq, np.ndarray):
                index_of_neuron = np.where(seq == neuron_id)[0][0]
            else:
                index_of_neuron = seq.index(neuron_id)
            neuron_indices.append(index_of_neuron)
            # Compute relative order (0 = first, 1 = last)
            relative_order = index_of_neuron / (len(seq) - 1) if len(seq) > 1 else 0
            relative_orders.append(relative_order)

print(f"Neuron ID: {neuron_id}")
print(f"Found in {len(active_seqs)} sequences")
print(f"Exact indices: {neuron_indices}")
print(f"Relative orders: {relative_orders}")

In [ ]:
n_neurons_keep_values = [10, 20, 30, 50, 75, 100, 150, 200, 350, 500]

# Store results for each R value and subsample
results_by_R = {}

for R in n_neurons_keep_values:
    neuron_indices_list = []
    relative_orders_list = []
    seq_lengths_list = []
    n_sequences_per_subsample = []
    
    # Run 50 subsamples for this R value
    for subsample in range(50):
        # Generate new downsampled sequences for this subsample
        seqs_downsampled, spk_times_downsampled, volume_downsampled = downsample_sequences(
            sequences, spk_times, volume, params["n_neurons"], 
            n_neurons_keep=R, min_length=5, random_state=subsample
        )
        
        # Get motif IDs

        motif_ids = [m for m in seqs_downsampled.keys() if len(seqs_downsampled[m]) > 20]
        #motif_ids = list(seqs_downsampled.keys())
        seqs_downsampled_filtered = {m: seqs_downsampled[m] for m in motif_ids if m in seqs_downsampled}
        
        if len(motif_ids) > 0:
            # Find the first valid neuron from the first non-empty sequence
            neuron_id = None
            for mid in motif_ids:
                if len(seqs_downsampled[mid]) > 0:
                    first_seq = seqs_downsampled[mid][0]
                    if len(first_seq) > 0:
                        neuron_id = first_seq[0]  # Get first neuron from first sequence
                        break
            
            if neuron_id is None:
                continue  # Skip this subsample if no valid neuron found
            
            seq_count = 0
            
            for motif_id, seq_list in seqs_downsampled.items():
                for seq in seq_list:
                    if neuron_id in seq:
                        seq_count += 1
                        seq_lengths_list.append(len(seq))
                        
                        # Find exact index
                        if isinstance(seq, np.ndarray):
                            index_of_neuron = np.where(seq == neuron_id)[0][0]
                        else:
                            index_of_neuron = seq.index(neuron_id)
                        
                        neuron_indices_list.append(index_of_neuron)
                        
                        # Compute relative order (0 = first, 1 = last)
                        relative_order = index_of_neuron / (len(seq) - 1) if len(seq) > 1 else 0
                        relative_orders_list.append(relative_order)
            
            n_sequences_per_subsample.append(seq_count)
    
    # Store aggregated results for this R
    results_by_R[R] = {
        'neuron_indices': neuron_indices_list,
        'relative_orders': relative_orders_list,
        'std_of_rel_orders': np.std(relative_orders_list) if relative_orders_list else 0,
        'seq_lengths': seq_lengths_list,
        'n_total_sequences': len(neuron_indices_list),
        'avg_seqs_per_subsample': np.mean(n_sequences_per_subsample) if n_sequences_per_subsample else 0
    }

In [ ]:
all_relative_orders = []
for R in n_neurons_keep_values:
    all_relative_orders.extend(results_by_R[R]['relative_orders'])

overall_mean_rel_order = np.mean(all_relative_orders) if all_relative_orders else 0
overall_std_rel_order = np.std(all_relative_orders) if all_relative_orders else 0

# Summary statistics
print(f"Analysis across different R values (50 subsamples each):")
print(f"{'R':<6} {'Total_seqs':<12} {'Avg/subsample':<14} {'Mean_idx':<10} {'Std_idx':<10} {'Std_rel_order':<15}")
print("-" * 70)
for R in n_neurons_keep_values:
    if results_by_R[R]['n_total_sequences'] > 0:
        mean_idx = np.mean(results_by_R[R]['neuron_indices'])
        std_idx = np.std(results_by_R[R]['neuron_indices'])
        std_rel = results_by_R[R]['std_of_rel_orders']
        print(f"{R:<6} {results_by_R[R]['n_total_sequences']:<12} {results_by_R[R]['avg_seqs_per_subsample']:<14.1f} {mean_idx:<10.2f} {std_idx:<10.2f} {std_rel:<15.3f}")
    else:
        print(f"{R:<6} {'0':<12} {'0':<14} {'N/A':<10} {'N/A':<10} {'N/A':<15}")

print("\n" + "=" * 70)
print(f"OVERALL AVERAGE RELATIVE ORDER (across all R values): {overall_mean_rel_order:.3f}")
print(f"OVERALL STANDARD DEVIATION RELATIVE ORDER (across all R values): {overall_std_rel_order:.3f}")
print("=" * 70)

with open(f"relative_orders_by_R_{params['sigma_range']}_{params['vol_param']}_{params['rho_sig']}.pkl", "wb") as f:
    pickle.dump(results_by_R, f)


### Sigma - Deviation of Each Neuron's Firing Order at R=100 in 50 Subsamples

In [ ]:
R = 100

# Dictionary to track ALL relative orders for EACH neuron across all 50 seeds
# Format: {neuron_id: [0.1, 0.15, 0.12, ...]}
orders_per_neuron = {} 

# Run 50 subsamples for this specific R value
for subsample in range(50):
    seqs_downsampled, _, _ = downsample_sequences(
        sequences, spk_times, volume, params["n_neurons"], 
        n_neurons_keep=R, min_length=5, random_state=subsample
    )
    
    # Filter motifs
    motif_ids = [m for m in seqs_downsampled.keys() if len(seqs_downsampled[m]) > 20]
    
    for motif_id in motif_ids:
        for seq in seqs_downsampled[motif_id]:
            # We can't calculate a relative order if there's only 1 neuron
            if len(seq) <= 1: 
                continue 
            
            # Iterate through the sequence and record EVERY neuron's relative order
            for index_of_neuron, neuron_id in enumerate(seq):
                relative_order = index_of_neuron / (len(seq) - 1)
                
                if neuron_id not in orders_per_neuron:
                    orders_per_neuron[neuron_id] = []
                orders_per_neuron[neuron_id].append(relative_order)

# Calculate the STD for EACH neuron individually
neuron_stds = []
for neuron_id, relative_orders in orders_per_neuron.items():
    # Only calculate STD if the neuron appeared enough times to have a meaningful variance
    if len(relative_orders) > 2:  
        neuron_stds.append(np.std(relative_orders))
        
# Finally, average all those individual neuron deviations together
if neuron_stds:
    final_avg_std = np.mean(neuron_stds)
else:
    final_avg_std = 0


In [ ]:
with open(f"neuron_stds_R_{R}_{params['sigma_range']}_{params['vol_param']}_{params['rho_sig']}.pkl", "wb") as f:
    pickle.dump({ "orders_per_neuron": orders_per_neuron, "neuron_stds": neuron_stds }, f)

### Summary statistics - sequence length

In [ ]:
# Visualize the downsampling effect
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 1. Sequences per motif
motif_ids = motif_ids_filtered
orig_counts = [len(sequences[m]) for m in motif_ids]
down_counts = [len(seqs_downsampled_filtered[m]) for m in motif_ids]

ax = axes[0, 0]
x = np.arange(len(motif_ids))
width = 0.35
ax.bar(x - width/2, orig_counts, width, label='Original', alpha=0.8)
ax.bar(x + width/2, down_counts, width, label='Downsampled (≤100 neurons)', alpha=0.8)
ax.set_xlabel('Motif ID')
ax.set_ylabel('Number of Sequences')
ax.set_title('Sequences per Motif: Original vs Downsampled')
ax.set_xticks(x)
ax.set_xticklabels(motif_ids, rotation=45)
ax.legend()
ax.grid(axis='y', alpha=0.3)

# 2. Average neurons per sequence
avg_orig_per_motif = [np.mean([len(s) for s in sequences[m]]) for m in motif_ids]
avg_down_per_motif = [np.mean([len(s) for s in seqs_downsampled_filtered[m]]) if seqs_downsampled_filtered[m] else 0 
                       for m in motif_ids]

ax = axes[0, 1]
ax.bar(x - width/2, avg_orig_per_motif, width, label='Original', alpha=0.8)
ax.bar(x + width/2, avg_down_per_motif, width, label='Downsampled', alpha=0.8)
ax.set_xlabel('Motif ID')
ax.set_ylabel('Average Neurons per Sequence')
ax.set_title('Average Sequence Length: Original vs Downsampled')
ax.set_xticks(x)
ax.set_xticklabels(motif_ids, rotation=45)
ax.legend()
ax.set_yscale('log')
ax.grid(axis='y', alpha=0.3)

# 3. Length distribution of original sequences
ax = axes[1, 0]
orig_lengths = [len(seq) for m in motif_ids for seq in sequences[m]]
mean_len = np.mean(orig_lengths)
ax.hist(orig_lengths, bins=20, alpha=0.7, color='steelblue', edgecolor='black')
ax.axvline(mean_len, color='red', linestyle='dashed', linewidth=1, label=f'Mean Length: {mean_len:.1f}')
ax.set_xlabel('Sequence Length (Number of Neurons)')
ax.set_ylabel('Count')
ax.set_title('Original Sequence Length Distribution')
ax.legend()
ax.grid(axis='y', alpha=0.3)

# 4. Length distribution of downsampled sequences
ax = axes[1, 1]
down_lengths = [len(seq) for m in motif_ids for seq in seqs_downsampled_filtered[m]]
mean_len_down = np.mean(down_lengths)
ax.hist(down_lengths, bins=20, alpha=0.7, color='coral', edgecolor='black')
ax.axvline(mean_len_down, color='red', linestyle='dashed', linewidth=1, label=f'Mean Length: {mean_len_down:.1f}')
ax.set_xlabel('Sequence Length (Number of Neurons)')
ax.set_ylabel('Count')
ax.set_title('Downsampled Sequence Length Distribution')
ax.legend()
ax.grid(axis='y', alpha=0.3)


plt.tight_layout()
plt.savefig(f'summary_{params['sigma_range']}_vol_{params['vol_param']}_volcorr_{params['rho_vol']}.jpg')
plt.savefig(f'summary_{params['sigma_range']}_vol_{params['vol_param']}_volcorr_{params['rho_vol']}.pdf')
plt.show()

In [ ]:
# Save number of seqeunces per motif after downsampling and filtering
with open(f'seq_counts_{params["sigma_range"]}_vol_{params["vol_param"]}_volcorr_{params["rho_vol"]}.pkl', 'wb') as f:
    pickle.dump({m: len(seqs_downsampled_filtered[m]) for m in motif_ids_filtered}, f)

In [ ]:
with open(f'seq_counts_(0.02, 0.4)_vol_(0.07, 0.9)_volcorr_0.0.pkl', 'rb') as f:
    data = pickle.load(f)
seq_counts_007 = data

with open(f'seq_counts_(0.02, 0.4)_vol_(0.07, 0.9)_volcorr_0.7.pkl', 'rb') as f:
    data = pickle.load(f)
seq_counts_007_07 = data

with open(f'seq_counts_(0.02, 0.4)_vol_(0.03, 0.9)_volcorr_0.0.pkl', 'rb') as f:
    data = pickle.load(f)
seq_counts_003 = data

with open(f'seq_counts_(1, 1)_vol_(0.07, 0.9)_volcorr_0.0.pkl', 'rb') as f:
    data = pickle.load(f)
seq_counts_11_007 = data

with open(f'seq_counts_(0.02, 0.4)_vol_(0.07, 0.5)_volcorr_0.0.pkl', 'rb') as f:
    data = pickle.load(f)
seq_counts_05 = data

## ADD ##
with open(f'seq_counts_(0.02, 0.4)_vol_(0.03, 0.9)_volcorr_0.7.pkl', 'rb') as f:
    data = pickle.load(f)
seq_counts_003_07 = data

# Get all unique keys
all_keys = sorted(set(seq_counts_007.keys()) | set(seq_counts_05.keys()) | set(seq_counts_007_07.keys()) | set(seq_counts_003.keys()))

# Sort all_keys by x_007 values in descending order
sorted_keys = sorted(all_keys, key=lambda k: seq_counts_007.get(k, 0), reverse=True)

In [ ]:
# Rank motifs within each dataset and plot grouped bars by rank
plt.figure(figsize=(10, 6))

# Ensure ranking helper exists in this execution context
if 'get_ranked_positions' not in globals():
    def get_ranked_positions(seq_dict):
        """Sort motifs by counts and return motifs and their rank positions (1-indexed)."""
        sorted_motifs = sorted(seq_dict.keys(), key=lambda k: seq_dict[k], reverse=True)
        ranks = {motif: i + 1 for i, motif in enumerate(sorted_motifs)}
        return sorted_motifs, ranks

# Dataset config: (data, color, label, offset)
width = 0.12
datasets = [
    (seq_counts_05, 'red',       'v=(0.07, 0.5), ρ=0.0', -2.5 * width),
    (seq_counts_11_007, 'gold',  'v=(0.07, 0.9), σ=(1,1)', -1.5 * width),
    (seq_counts_007, 'steelblue','v=(0.07, 0.9), ρ=0.0', -0.5 * width),
    (seq_counts_007_07, 'purple','v=(0.07, 0.9), ρ=0.7', 0.5 * width),
    (seq_counts_003, 'coral',    'v=(0.03, 0.9), ρ=0.0', 1.5 * width),
    (seq_counts_003_07, 'seagreen','v=(0.03, 0.9), ρ=0.7', 2.5 * width),
]

# Rank motifs independently in each dataset
ranked = []
max_rank = 0
for seq_counts, color, label, offset in datasets:
    motifs, ranks = get_ranked_positions(seq_counts)
    x_rank = np.array([ranks[m] for m in motifs], dtype=float)
    y_val = [seq_counts[m] for m in motifs]
    ranked.append((x_rank, y_val, color, label, offset))
    max_rank = max(max_rank, len(motifs))

# Plot side-by-side bars at each rank index
for x_rank, y_val, color, label, offset in ranked:
    plt.bar(x_rank + offset, y_val, width=width, color=color, edgecolor='black', alpha=0.7, label=label)

plt.xlabel('Motif Rank within each parameter set (1 = highest sequence count)')
plt.ylabel('Number of Sequences (Downsampled)', fontsize=13)
plt.title('Ranked Motif Sequence Counts Across Parameter Sets\nσ=(0.02, 0.4) | filter motifs with sequence count>20', fontsize=15)
plt.xticks(np.arange(1, max_rank + 1))
plt.legend(title='Parameter Sets', loc='lower right', fontsize=8)
plt.grid(axis='y', alpha=0.3)
plt.tick_params(axis='x', labelsize=10)
plt.tick_params(axis='y', labelsize=10)
plt.tight_layout()
plt.savefig('w(1,1)seq_counts_comparison.jpg', dpi=300, bbox_inches='tight')
plt.savefig('w(1,1)seq_counts_comparison.pdf', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# Sort each dataset independently and assign ranks
def get_ranked_positions(seq_dict):
    """Sort motifs by counts and return motifs and their rank positions (1-indexed)"""
    sorted_motifs = sorted(seq_dict.keys(), key=lambda k: seq_dict[k], reverse=True)
    ranks = {motif: i + 1 for i, motif in enumerate(sorted_motifs)}
    return sorted_motifs, ranks

# Get independent rankings for each dataset
motifs_05, ranks_05 = get_ranked_positions(seq_counts_05)
motifs_007, ranks_007 = get_ranked_positions(seq_counts_007)
motifs_007_07, ranks_007_07 = get_ranked_positions(seq_counts_007_07)
motifs_003, ranks_003 = get_ranked_positions(seq_counts_003)
motifs_003_07, ranks_003_07 = get_ranked_positions(seq_counts_003_07)

# Create plot
plt.figure(figsize=(10, 5))

# Plot each dataset with its own independent ranking
x_pos_05 = [ranks_05[k] for k in motifs_05]
y_val_05 = [seq_counts_05[k] for k in motifs_05]
plt.scatter(x_pos_05, y_val_05, color='red', edgecolor='black', alpha=0.6, s=80, label='v=(0.07, 0.5), ρ=0.0')

x_pos_007_07 = [ranks_007_07[k] for k in motifs_007_07]
y_val_007_07 = [seq_counts_007_07[k] for k in motifs_007_07]
plt.scatter(x_pos_007_07, y_val_007_07, color='purple', edgecolor='black', alpha=0.7, s=80, label='v=(0.07, 0.9), ρ=0.7')

x_pos_007 = [ranks_007[k] for k in motifs_007]
y_val_007 = [seq_counts_007[k] for k in motifs_007]
plt.scatter(x_pos_007, y_val_007, color='steelblue', edgecolor='black', alpha=0.7, s=80, label='v=(0.07, 0.9), ρ=0.0')

x_pos_003_07 = [ranks_003_07[k] for k in motifs_003_07]
y_val_003_07 = [seq_counts_003_07[k] for k in motifs_003_07]
plt.scatter(x_pos_003_07, y_val_003_07, color='seagreen', edgecolor='black', alpha=0.7, s=80, label='v=(0.03, 0.9), ρ=0.7')

x_pos_003 = [ranks_003[k] for k in motifs_003]
y_val_003 = [seq_counts_003[k] for k in motifs_003]
plt.scatter(x_pos_003, y_val_003, color='coral', edgecolor='black', alpha=0.7, s=80, label='v=(0.03, 0.9), ρ=0.0')


plt.xlabel('Motif Rank (1=highest, 20=lowest sequence count for each dataset)', fontsize=13)
plt.ylabel('Number of Sequences after Downsampling', fontsize=13)   
plt.title('Number of Sequences per Motif Across Different Volume Parameters\nσ=(0.02, 0.4) | filter motifs with sequence count>20', fontsize=15)
plt.tick_params(axis='x', labelsize=10)
plt.tick_params(axis='y', labelsize=10)
plt.xticks(range(1, 21))
plt.legend(title='Parameter Sets', loc='lower right', fontsize=8)
plt.grid(axis='both', alpha=0.3)
plt.tight_layout()
plt.savefig('seq_counts_comparison_scatter.jpg', dpi=300, bbox_inches='tight')
plt.savefig('seq_counts_comparison_scatter.pdf', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
print(f'Total motifs with 100 sequences in (0.03, 0.9), ρ=0.7: {sum(1 for k in seq_counts_003_07 if seq_counts_003_07[k] == 100)}')
print(f'Motifs with 100 sequences in (0.03, 0.9), ρ=0.7: {[k for k in seq_counts_003_07 if seq_counts_003_07[k] == 100]}')
print(f'Total motifs with >50 sequences in (0.03, 0.9), ρ=0.7: {sum(1 for k in seq_counts_003_07 if seq_counts_003_07[k] > 50)}')
print(f'Motifs with >50 sequences in (0.03, 0.9), ρ=0.7: {[k for k in seq_counts_003_07 if seq_counts_003_07[k] > 50]}')

### Motif and Sequence Retention per R

In [ ]:
# Test different n_neurons_keep values and count motifs (with min_length=5 filter)
n_neurons_keep_values = [10, 20, 30, 50, 100, 150, 200, 500]
motif_counts_filtered = []
seqs_retained_pct_list = []

print("Testing different n_neurons_keep values (with min_length=5)...\n")
for n_keep in n_neurons_keep_values:
    seqs_down, _, _ = downsample_sequences(sequences, spk_times, volume, params['n_neurons'], n_keep, min_length=5, random_state=1)
    
    # Filter motifs with more than 20 sequences
    motif_ids = [m for m in seqs_down.keys() if len(seqs_down[m]) > 20]
    motifs_with_seqs = len(motif_ids)
    motif_counts_filtered.append(motifs_with_seqs)
    
    # Number of sequences retained
    seqs_retained = sum(len(seqs_down[m]) for m in motif_ids)
    total_orig = sum(len(seqs) for seqs in sequences.values())
    seqs_retained_pct = seqs_retained / total_orig * 100
    seqs_retained_pct_list.append(seqs_retained_pct)

    print(f"R={n_keep:5d} → {motifs_with_seqs} motifs retained seqs retained: {seqs_retained} ({seqs_retained_pct:.1f}%)")

# Plot with dual y-axes
fig, ax1 = plt.subplots(figsize=(14, 6))

# Left y-axis: Motifs retained
color1 = 'coral'
ax1.set_xlabel('R', fontsize=12)
ax1.set_ylabel('Number of Motifs Retained', color=color1, fontsize=12)
ax1.plot(n_neurons_keep_values, motif_counts_filtered, marker='s', linewidth=2, 
         markersize=8, color=color1, label='Motifs Retained')
ax1.tick_params(axis='y', labelcolor=color1)
ax1.set_xticks(n_neurons_keep_values)
ax1.set_xticklabels(n_neurons_keep_values)
ax1.set_ylim([0, 22])
ax1.set_yticks(range(0, max(motif_counts_filtered) + 1)) # Set y-ticks based on actual motif counts
ax1.grid(True, alpha=0.3, axis='y')

# Right y-axis: Sequences retained percentage
ax2 = ax1.twinx()
color2 = 'steelblue'
ax2.set_ylabel('Sequences Retained (%)', color=color2, fontsize=12)
ax2.plot(n_neurons_keep_values, seqs_retained_pct_list, marker='o', linewidth=2, 
         markersize=8, color=color2, label='Sequences Retained %')
ax2.tick_params(axis='y', labelcolor=color2)
ax2.set_yticks(range(0, 101, 10))  # Set y-ticks for right axis at intervals of 10%

# Set values for right y-axis dots
for i, pct in enumerate(seqs_retained_pct_list):
    ax2.text(n_neurons_keep_values[i], pct + 2, f"{pct:.1f}%", color=color2, fontsize=8, ha='center')

ax1.set_title(f'Motif & Sequence Retention vs Neuron Threshold\nsigma: {params["sigma_range"]} | volume: {params["vol_param"]}', fontsize=12)


plt.tight_layout()
plt.savefig(f'motifnum_{params["sigma_range"]}_vol_{params["vol_param"]}_volcorr_{params["rho_vol"]}.jpg')
plt.savefig(f'motifnum_{params["sigma_range"]}_vol_{params["vol_param"]}_volcorr_{params["rho_vol"]}.pdf')
plt.show()

### Mean Retention of Sequences and Motifs of 50 Subsample

In [ ]:
n_neurons_keep_values = [10, 20, 30, 50, 75, 100, 150, 200, 350, 500]
seed_range = range(0, 50)  # Seeds from 0 to 49
# Dictionary to store results: {seed: {n_keep: (motifs_retained, seqs_retained_pct)}}
results_per_seed = {}

# Compute mean retention metrics across seeds
for seed in seed_range:
    print(f"Processing seed={seed}...")
    motif_counts = []
    seqs_retained_pcts = []
    
    for n_keep in n_neurons_keep_values:
        # Downsample with this seed
        seqs_down, _, _ = downsample_sequences(sequences, spk_times, volume, 
                                               params["n_neurons"], n_neurons_keep=n_keep, 
                                               min_length=5, random_state=seed)
        
        # Filter motifs with more than 20 sequences
        motif_ids = [m for m in seqs_down.keys() if len(seqs_down[m]) > 20]
        motifs_with_seqs = len(motif_ids)
        motif_counts.append(motifs_with_seqs)
        
        # Calculate sequence retention percentage
        seqs_retained = sum(len(seqs_down[m]) for m in motif_ids)
        total_orig = sum(len(seqs) for seqs in sequences.values())
        seqs_retained_pct = seqs_retained / total_orig * 100
        seqs_retained_pcts.append(seqs_retained_pct)
    
    results_per_seed[seed] = {
        'motif_counts': motif_counts,
        'seqs_retained_pcts': seqs_retained_pcts
    }

# Compute mean retention metrics across seeds
mean_motif_counts = [sum(results_per_seed[seed]['motif_counts'][i] for seed in seed_range) / len(seed_range) for i in range(len(n_neurons_keep_values))]
mean_seqs_retained_pcts = [sum(results_per_seed[seed]['seqs_retained_pcts'][i] for seed in seed_range) / len(seed_range) for i in range(len(n_neurons_keep_values))]
std_motif_counts = [np.std([results_per_seed[seed]['motif_counts'][i] for seed in seed_range]) for i in range(len(n_neurons_keep_values))]
std_seqs_retained_pcts = [np.std([results_per_seed[seed]['seqs_retained_pcts'][i] for seed in seed_range]) for i in range(len(n_neurons_keep_values))]

# Create plot with dual y-axes
fig, ax1 = plt.subplots(figsize=(14, 6))
color1 = 'coral'
ax1.errorbar(n_neurons_keep_values, mean_motif_counts, yerr=std_motif_counts, marker='s', linewidth=2, 
             markersize=8, color=color1, label=f'Seed {seed} (Motifs)', alpha=0.8)
    
    # Right axis: Sequences retained percentage (already on same axis, just different y)
    # We'll add to ax2 next

ax1.set_xlabel('R', fontsize=12)
ax1.set_ylabel('Number of Motifs Retained', fontsize=12, color=color1)
ax1.set_xticks(n_neurons_keep_values)
ax1.set_ylim([0, 22])
ax1.grid(True, alpha=0.3, axis='y')
ax1.set_yticks(range(0, int(max(mean_motif_counts)) + 1))
ax1.tick_params(axis='y', labelcolor=color1)

# Create second y-axis for sequences retained
ax2 = ax1.twinx()
color2 = 'steelblue'
ax2.errorbar(n_neurons_keep_values, mean_seqs_retained_pcts, yerr=std_seqs_retained_pcts, marker='o', linewidth=2, 
             markersize=8, color=color2, label=f'Seed {seed} (Seqs %)', 
             linestyle='--', alpha=0.8)

ax2.set_ylabel('Sequences Retained (%)', fontsize=12, color=color2)
ax2.set_yticks(range(0, 101, 10))
ax2.tick_params(axis='y', labelcolor=color2)

ax1.set_title(f'Motif & Sequence Retention vs Neuron Threshold (Multiple Seeds)\nsigma: {params["sigma_range"]} | volume: {params["vol_param"]}', 
              fontsize=12)

plt.tight_layout()
plt.savefig(f'meanretention_multiple_seeds_{params["sigma_range"]}_vol_{params["vol_param"]}_volcorr_{params["rho_vol"]}.jpg')
plt.savefig(f'meanretention_multiple_seeds_{params["sigma_range"]}_vol_{params["vol_param"]}_volcorr_{params["rho_vol"]}.pdf')
plt.show()

In [ ]:
# Convert dict to flat list for allmot
# Filter motifs with more than 20 sequences (to be consistent with earlier filtering)
motif_ids_for_allmot = [m for m in seqs_downsampled_filtered.keys() 
                        if len(seqs_downsampled_filtered[m]) > 20]

seqs_flat = []
for motif_id in motif_ids_for_allmot:
    seqs_flat.extend(seqs_downsampled_filtered[motif_id])

print(f"Total sequences (flat): {len(seqs_flat)}")
print(f"Expected from filtered: {sum(len(seqs) for seqs in seqs_downsampled_filtered.values())}")

# Now run allmot
rep_index_down, nsig_down, pval_down, bmat_down, zmat_down, corrmat_down = rs.allmot(seqs_flat, nrm)
print("Done!")
print("Significant motifs found:", motif_ids_for_allmot)

### Subsampled Correlation Matrix

In [ ]:
plt.figure(figsize=(7, 5))
plt.imshow(corrmat_down, aspect='auto', cmap='RdBu_r')
plt.colorbar(label='Correlation')
plt.title('Pairwise Sequences Correlation Matrix')
plt.xlabel('Sequence Index')
plt.ylabel('Sequence Index')    
plt.suptitle(f'Correlation Matrix Downsampled| sigma: {params["sigma_range"]} | volume: {params["vol_param"]}', fontsize=14)
#plt.savefig(f'origcorrmat_{params["sigma_range"]}_vol_{params["vol_param"]}_volcorr_{params["rho_sig"]}.jpg')
#plt.savefig(f'origcorrmat_{params["sigma_range"]}_vol_{params["vol_param"]}_volcorr_{params["rho_sig"]}.pdf')
plt.show()

### Original Simulation Correlation Matrix

In [ ]:
# Now run allmot
rep_index, nsig, pval, bmat, zmat, corrmat = rs.allmot(seqs, nrm)

plt.figure(figsize=(7, 5))
plt.imshow(corrmat, aspect='auto', cmap='RdBu_r')
plt.colorbar(label='Correlation')
plt.title('Pairwise Sequences Correlation Matrix')
plt.xlabel('Sequence Index')
plt.ylabel('Sequence Index')    
plt.suptitle(f'Correlation Matrix | sigma: {params["sigma_range"]} | volume: {params["vol_param"]}', fontsize=14)
plt.savefig(f'origcorrmat_seed1_{params["sigma_range"]}_vol_{params["vol_param"]}_volcorr_{params["rho_sig"]}.jpg')
plt.savefig(f'origcorrmat_seed1_{params["sigma_range"]}_vol_{params["vol_param"]}_volcorr_{params["rho_sig"]}.pdf')
plt.show()

### Correlation matrices from different seed points

In [ ]:
# subplot of correlation matrices of different seed points for downsampled data
fig, axes = plt.subplots(2, 2, figsize=(14, 12))
seed_points = [0,1,2,3]
for i, seed in enumerate(seed_points):
    seqs_down, _, _ = downsample_sequences(sequences, spk_times, volume, params['n_neurons'], n_neurons_keep=100, min_length=5, random_state=seed)
    
    # Filter motifs with more than 20 sequences
    motif_ids = [m for m in seqs_down.keys() if len(seqs_down[m]) > 20]
    seqs_flat = []
    for motif_id in motif_ids:
        seqs_flat.extend(seqs_down[motif_id])
    
    rep_index_down, nsig_down, pval_down, bmat_down, zmat_down, corrmat_down = rs.allmot(seqs_flat, nrm)
    
    ax = axes[i//2, i%2]
    im = ax.imshow(corrmat_down, aspect='auto', cmap='RdBu_r')
    ax.set_title(f'Seed: {seed} | Motifs: {len(motif_ids)} | Seqs: {len(seqs_flat)}', fontsize=12)
    ax.set_xlabel('Sequence Index')
    ax.set_ylabel('Sequence Index')
    fig.colorbar(im, ax=ax) 
plt.suptitle(f'Correlation Matrices for Different Seeds (Downsampled) | sigma: {params["sigma_range"]} | volume: {params["vol_param"]}', fontsize=14)
plt.tight_layout(rect=[0, 0.03, 1, 0.95])
plt.savefig(f'corrmat_seeds_{params['sigma_range']}_vol_{params['vol_param']}_volcorr_{params['rho_vol']}.jpg')
plt.savefig(f'corrmat_seeds_{params['sigma_range']}_vol_{params['vol_param']}_volcorr_{params['rho_vol']}.pdf')
plt.show()  

### Subsampled Volume Distribution

In [ ]:
# Plot downsampled volume distribution (filtered >20 sequences)
fig, ax = plt.subplots(figsize=(5,4))

# flatten the volumes from filtered sequences for histogram plotting
flattened_volumes = []

motif_ids_filtered = [m for m in sequences.keys()]

for motif_id in motif_ids_filtered:
    flattened_volumes.extend(volume[motif_id])
ax.hist(flattened_volumes, bins=30, color='steelblue', edgecolor='black', alpha=0.7)
ax.set_xlabel('Volume', fontsize=12)
ax.set_ylabel('Count', fontsize=12)
ax.set_title(f'Volume Distribution of N=100k\nσ: {params["sigma_range"]} | v: {params["vol_param"]}', fontsize=14)
plt.tight_layout()
plt.savefig(f'NOFILTERvolume_dist_{params['sigma_range']}_vol_{params['vol_param']}_volcorr_{params['rho_vol']}.jpg')
plt.savefig(f'NOFILTERvolume_dist_{params['sigma_range']}_vol_{params['vol_param']}_volcorr_{params['rho_vol']}.pdf')
plt.show()

#with open(f'median_volume_100_{params["sigma_range"]}_vol_{params["vol_param"]}_volcorr_{params["rho_vol"]}.pkl', 'wb') as f:
#    pickle.dump({m: np.median(volume_downsampled_filt[m]) for m in motif_ids_for_allmot}, f)

### Volume Distribution of 50 Subsamples at R = 100 ###

In [ ]:
# Histogram of active-neuron volume distribution across 50 subsamples at n=100

n_keep = 100
n_seeds = 50
min_seqs_per_motif = 20   # set to 0 if you do not want motif filtering

all_volumes = []
for seed in range(n_seeds):
    seqs_down, _, _ = downsample_sequences(
        sequences, spk_times, volume, params["n_neurons"],
        n_neurons_keep=n_keep, min_length=5, random_state=seed
    )

    # Optional motif filter
    motif_ids = [m for m in seqs_down if len(seqs_down[m]) > min_seqs_per_motif]

    for motif_id in motif_ids:
        for seq in seqs_down[motif_id]:
            if len(seq) == 0:
                continue
            seq_idx = np.asarray(seq, dtype=int)
            # volume shape: (n_neurons, n_motifs)
            all_volumes.extend(volume[seq_idx, motif_id])

all_volumes = np.asarray(all_volumes, dtype=float)
median_volume = np.median(all_volumes) if all_volumes.size else np.nan

fig, ax = plt.subplots(figsize=(8, 6))
n, bins, patches = ax.hist(all_volumes, bins=30, color="steelblue", edgecolor="black", alpha=0.7)

# Optional count labels
for count, patch in zip(n, patches):
    if count > 0:
        ax.text(
            patch.get_x() + patch.get_width() / 2,
            patch.get_height(),
            f"{int(count)}",
            ha="center",
            va="bottom",
            fontsize=7,
            rotation=45
        )

ax.axvline(median_volume, color="red", linestyle="--", linewidth=2, label=f"Median: {median_volume:.3f}")
ax.set_xlabel("Volume", fontsize=12)
ax.set_ylabel("Count", fontsize=12)
ax.set_title(
    f"Volume Distribution Across {n_seeds} Random Subsamples for R={n_keep}\n"
    f"sigma: {params['sigma_range']} | volume: {params['vol_param']}",
    fontsize=14
)
ax.legend()
plt.tight_layout()
plt.show()

print(f"Median volume (R={n_keep}, {n_seeds} seeds): {median_volume:.6f}")

In [ ]:
# Histogram of active-neuron volume distributions across parameter settings
min_seqs_per_motif = 20   # set to 0 if you do not want motif filtering

dataset_specs = [
    (sequences_05, volume_05, 'v=(0.07, 0.5), ρ=0.0'),
    (sequences, volume, 'v=(0.07, 0.9), ρ=0.0'),
    (sequences_003, volume_003, 'v=(0.03, 0.9), ρ=0.0'),
    (sequences_003_07, volume_003_07, 'v=(0.03, 0.9), ρ=0.7'),
]

fig, axes = plt.subplots(2, 2, figsize=(14, 12), sharex=True, sharey=True)
axes = axes.flatten()

for ax, (seq_dict, vol_matrix, label) in zip(axes, dataset_specs):
    active_volumes = []
    motif_ids = [m for m in seq_dict.keys() if len(seq_dict[m]) > min_seqs_per_motif]

    for motif_id in motif_ids:
        for seq in seq_dict[motif_id]:
            if len(seq) == 0:
                continue
            seq_idx = np.asarray(seq, dtype=int)
            active_volumes.extend(vol_matrix[seq_idx, motif_id])

    active_volumes = np.asarray(active_volumes, dtype=float)
    active_volumes = active_volumes[np.isfinite(active_volumes)]
    median_volume = np.median(active_volumes) if active_volumes.size else np.nan

    ax.hist(active_volumes, bins=30, color="steelblue", edgecolor="black", alpha=0.7)
    if np.isfinite(median_volume):
        ax.axvline(median_volume, color="red", linestyle="--", linewidth=2, label=f"Median: {median_volume:.3f}")
    ax.set_title(label, fontsize=12)
    ax.set_xlabel("Volume", fontsize=12)
    ax.set_ylabel("Count", fontsize=12)
    ax.legend()

fig.suptitle(
    f"Active-Neuron Volume Distribution Across Parameter Settings\n"
    f"sigma: {params['sigma_range']} | volume: {params['vol_param']}",


In [ ]:
 # get mean volume of downsampled sequences compared to original per motif and plot in one plot
# volume is shape (n_neurons, n_motifs)
# For each motif, compute the mean volume across all neurons that appear in sequences for that motif

motif_ids_filtered = [m for m in seqs_downsampled_filtered.keys() if len(seqs_downsampled_filtered[m]) > 20]

mean_volume_orig = {
    m: np.mean([volume[neuron, m] for seq in sequences[m] for neuron in seq])
    for m in motif_ids_filtered
}

mean_volume_down = {
    m: np.mean([volume[neuron, m] for seq in seqs_downsampled_filtered[m] for neuron in seq])
    if seqs_downsampled_filtered[m] else 0
    for m in motif_ids_filtered
}

plt.figure(figsize=(10, 6))
# Sort by highest downsampled mean first
motif_ids = motif_ids_filtered
sorted_indices = np.argsort([mean_volume_down[m] for m in motif_ids])[::-1] # Descending order
motif_ids_sorted = [motif_ids[i] for i in sorted_indices]

x = np.arange(len(motif_ids_sorted))
width = 0.35
plt.bar(x - width/2, [mean_volume_orig[m] for m in motif_ids_sorted], width, label='Original', alpha=0.8)
plt.bar(x + width/2, [mean_volume_down[m] for m in motif_ids_sorted], width, label='Downsampled', alpha=0.8)
plt.xlabel('Motif ID')
plt.xticks(x, motif_ids_sorted)
plt.ylabel('Mean Volume')
plt.title(f'Mean Volume per Motif: Original vs Downsampled (sorted by downsampled)\nsigma: {params["sigma_range"]} | volume: {params["vol_param"]}', fontsize=12)
plt.legend()
plt.tight_layout()
plt.savefig(f'mean_volume_{params["sigma_range"]}_vol_{params["vol_param"]}_volcorr_{params["rho_vol"]}.jpg')
plt.savefig(f'mean_volume_{params["sigma_range"]}_vol_{params["vol_param"]}_volcorr_{params["rho_vol"]}.pdf')
plt.show() 

In [ ]:
# Compute median of valume for one subsample at R=100 and save as a variable


### Mean Correlation per Motif (sorted)

In [ ]:
with open(f'downsample_R100_N100000_sig_{params["sigma_range"]}_vol_{params["vol_param"]}_rhosig_{params["rho_sig"]}_rhovol_{params["rho_vol"]}.pkl', 'rb') as f:
    data = pickle.load(f)
    
seqs_downsampled_filtered = data["seqs_down"]
volume_downsampled_filt = data["volume_down"] 

motif_ids_for_allmot = [m for m in seqs_downsampled_filtered.keys() 
                        if len(seqs_downsampled_filtered[m]) > 20]

seqs_flat = []
for motif_id in motif_ids_for_allmot:
    seqs_flat.extend(seqs_downsampled_filtered[motif_id])

print(f"Total sequences (flat): {len(seqs_flat)}")
print(f"Expected from filtered: {sum(len(seqs) for seqs in seqs_downsampled_filtered.values())}")

# Now run allmot
rep_index_down, nsig_down, pval_down, bmat_down, zmat_down, corrmat_down = rs.allmot(seqs_flat, nrm)

In [ ]:
# Create motif index array to track which motif each sequence belongs to
motif_index_down = []

for motif_id in motif_ids_for_allmot:
    motif_index_down.extend([motif_id] * len(seqs_downsampled_filtered[motif_id]))
motif_index_down = np.array(motif_index_down)

# Compute mean correlation per motif using the imported function
within_score, within_var = add_within_clust_score(motif_index_down, zmat_down, bmat_down) ######### CHANGED INTO ZMAT !!!!!

# Create proper mapping from unique motif IDs to scores
unique_motifs = np.unique(motif_index_down)
motif_to_score = {motif: score for motif, score in zip(unique_motifs, within_score)}
motif_to_var = {motif: var for motif, var in zip(unique_motifs, within_var)}

# Sort by highest score first (excluding NaN)
valid_motifs = [m for m in unique_motifs if not np.isnan(motif_to_score[m])]
sorted_motifs = sorted(valid_motifs, key=lambda m: motif_to_score[m], reverse=True)
scores_sorted = [motif_to_score[m] for m in sorted_motifs]
vars_sorted = [motif_to_var[m] for m in sorted_motifs]

with open(f'mean_correlation_scores_{params["sigma_range"]}_{params["vol_param"]}_{params["rho_sig"]}.pkl', 'wb') as f:
    pickle.dump({'motifs': sorted_motifs, 'scores': scores_sorted, 'variances': vars_sorted}, f)


In [ ]:
# Plot
fig, ax = plt.subplots(figsize=(14, 6))
ax.errorbar(range(len(sorted_motifs)), scores_sorted, yerr=vars_sorted, fmt='o', color='steelblue', ecolor='black', capsize=5)
ax.set_xlabel('Motif ID', fontsize=12)
ax.set_ylabel('Mean Correlation Score', fontsize=12)
ax.set_title(f'Mean Correlation Score per Motif (Downsampled, sorted by score)\nsigma: {params["sigma_range"]} | volume: {params["vol_param"]}', fontsize=12)
ax.set_xticks(range(len(sorted_motifs)))
ax.set_xticklabels(sorted_motifs)
ax.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.savefig(f'mean_corr_score_{params["sigma_range"]}_vol_{params["vol_param"]}__{params["rho_vol"]}.jpg')
plt.savefig(f'mean_corr_score_{params["sigma_range"]}_vol_{params["vol_param"]}_{params["rho_vol"]}.pdf')
plt.show()

In [ ]:
with open(f'mean_correlation_scores_(0.02, 0.4)_(0.07, 0.9)_0.0.pkl', 'rb') as f:
    data_00204 = pickle.load(f)

with open(f'mean_correlation_scores_(0.02, 0.04)_(0.07, 0.9)_0.0.pkl', 'rb') as f:
    data_002004 = pickle.load(f)

with open(f'mean_correlation_scores_(0.4, 0.4)_(0.07, 0.9)_0.0.pkl', 'rb') as f:
    data_0404 = pickle.load(f)   

with open(f'mean_correlation_scores_(1, 1)_(0.07, 0.9)_0.0.pkl', 'rb') as f:
    data_11 = pickle.load(f)

with open(f'mean_correlation_scores_(0.02, 0.4)_(0.03, 0.9)_0.0.pkl', 'rb') as f:
    data_00204_003 = pickle.load(f)

with open(f'mean_correlation_scores_(0.02, 0.4)_(0.07, 0.5)_0.0.pkl', 'rb') as f:
    data_00204_05 = pickle.load(f)

with open(f'mean_correlation_scores_(0.02, 0.4)_(0.03, 0.9)_0.7.pkl', 'rb') as f:
    data_00204_003_07 = pickle.load(f)

with open(f'mean_correlation_scores_(1, 1)_(0.07, 0.9)_0.7.pkl', 'rb') as f:
    data_11_07 = pickle.load(f)

with open(f'mean_correlation_scores_(1, 1)_(0.07, 0.5)_0.0.pkl', 'rb') as f:
    data_11_05 = pickle.load(f)

with open(f'mean_correlation_scores_(1, 1)_(0.1, 0.5)_0.0.pkl', 'rb') as f:
    data_11_0105 = pickle.load(f)

In [ ]:
#plot mean correlation scores by ranking motifs by their scores independently for each dataset
descending_00204 = sorted(zip(data_00204['motifs'], data_00204['scores'], data_00204['variances']), key=lambda x: x[1], reverse=True)
descending_002004 = sorted(zip(data_002004['motifs'], data_002004['scores'], data_002004['variances']), key=lambda x: x[1], reverse=True)
descending_0404 = sorted(zip(data_0404['motifs'], data_0404['scores'], data_0404['variances']), key=lambda x: x[1], reverse=True)
descending_11 = sorted(zip(data_11['motifs'], data_11['scores'], data_11['variances']), key=lambda x: x[1], reverse=True)
descending_11_07 = sorted(zip(data_11_07['motifs'], data_11_07['scores'], data_11_07['variances']), key=lambda x: x[1], reverse=True)

fig, ax = plt.subplots(figsize=(10, 6))
ax.errorbar(range(len([s for m, s, v in descending_002004])), [s for m, s, v in descending_002004], yerr=[v for m, s, v in descending_002004], fmt='o', capsize=5, label='(0.02, 0.04)', color='steelblue')
ax.errorbar(range(len([s for m, s, v in descending_00204])), [s for m, s, v in descending_00204], yerr=[v for m, s, v in descending_00204], fmt='o', capsize=5, label='(0.02, 0.4)', color='coral')
ax.errorbar(range(len([s for m, s, v in descending_0404])), [s for m, s, v in descending_0404], yerr=[v for m, s, v in descending_0404], fmt='o', capsize=5, label='(0.4, 0.4)', color='purple')
ax.errorbar(range(len([s for m, s, v in descending_11])), [s for m, s, v in descending_11], yerr=[v for m, s, v in descending_11], fmt='o', capsize=5, label='(1.1, 1.1)', color='lightgreen')
ax.errorbar(range(len([s for m, s, v in descending_11_07])), [s for m, s, v in descending_11_07], yerr=[v for m, s, v in descending_11_07], fmt='o', capsize=5, label='(1.1, 1.1), ρ = 0.7', color='gold')
ax.set_xticks(np.arange(20))          
ax.set_xticklabels(np.arange(1,21))     
ax.set_xlim(-0.5, 19.5)  
ax.set_xlabel('Motif Rank', fontsize=15)
ax.set_ylabel('Mean Correlation Score (Z-Score)', fontsize=15)
ax.tick_params(axis='x', labelsize=9)
ax.tick_params(axis='y', labelsize=10)
ax.set_title(f'Mean Correlation Scores per Motif of One Subsample (Independently Ranked)\nvolume: {params["vol_param"]}', fontsize=15)
ax.legend(title='σ Range', fontsize=9)
plt.tight_layout()
plt.savefig(f'mean_corr_zscores_independent_ranking_{params["vol_param"]}.jpg', dpi=300)
plt.savefig(f'mean_corr_zscores_independent_ranking_{params["vol_param"]}.pdf', dpi=300)
plt.show()

In [ ]:
sorted_indices = np.argsort(data_00204['scores'])[::-1]
sorted_motifs = [data_00204['motifs'][i] for i in sorted_indices]
sorted_scores_00204 = [data_00204['scores'][i] for i in sorted_indices]
sorted_vars_00204 = [data_00204['variances'][i] for i in sorted_indices]

# Create mapping from motif to sorted position
motif_to_pos = {motif: i for i, motif in enumerate(sorted_motifs)}

# Plot comparison of mean correlation scores across different parameters
fig, ax = plt.subplots(figsize=(12, 6))
x_pos = np.arange(len(sorted_motifs))

# Plot sigma=(0.02, 0.4) at the sorted positions
ax.errorbar(x_pos, sorted_scores_00204, yerr=sorted_vars_00204, fmt='o', color='salmon', ecolor='salmon', capsize=5, label='* sigma=(0.02, 0.4)')

# Plot other datasets at matching motif positions
x_002004 = [motif_to_pos[m] for m in data_002004['motifs'] if m in motif_to_pos]
scores_002004 = [data_002004['scores'][i] for i, m in enumerate(data_002004['motifs']) if m in motif_to_pos]
vars_002004 = [data_002004['variances'][i] for i, m in enumerate(data_002004['motifs']) if m in motif_to_pos]
ax.errorbar(x_002004, scores_002004, yerr=vars_002004, fmt='o', color='steelblue', ecolor='steelblue', capsize=5, label='sigma=(0.02, 0.04)')

x_0404 = [motif_to_pos[m] for m in data_0404['motifs'] if m in motif_to_pos]
scores_0404 = [data_0404['scores'][i] for i, m in enumerate(data_0404['motifs']) if m in motif_to_pos]
vars_0404 = [data_0404['variances'][i] for i, m in enumerate(data_0404['motifs']) if m in motif_to_pos]
ax.errorbar(x_0404, scores_0404, yerr=vars_0404, fmt='o', color='orchid', ecolor='orchid', capsize=5, label='sigma=(0.4, 0.4)')

x_11 = [motif_to_pos[m] for m in data_11['motifs'] if m in motif_to_pos]
scores_11 = [data_11['scores'][i] for i, m in enumerate(data_11['motifs']) if m in motif_to_pos]
vars_11 = [data_11['variances'][i] for i, m in enumerate(data_11['motifs']) if m in motif_to_pos]
ax.errorbar(x_11, scores_11, yerr=vars_11, fmt='o', color='lightgreen', ecolor='lightgreen', capsize=5, label='sigma=(1, 1)')

x_11_07 = [motif_to_pos[m] for m in data_11_07['motifs'] if m in motif_to_pos]
scores_11_07 = [data_11_07['scores'][i] for i, m in enumerate(data_11_07['motifs']) if m in motif_to_pos]
vars_11_07 = [data_11_07['variances'][i] for i, m in enumerate(data_11_07['motifs']) if m in motif_to_pos]
ax.errorbar(x_11_07, scores_11_07, yerr=vars_11_07, fmt='o', color='gold', ecolor='gold', capsize=5, label='sigma=(1, 1), ρ=0.7)')

ax.set_xlabel('Motif ID', fontsize=12)
ax.set_ylabel('Mean Correlation Score', fontsize=12)
ax.set_title('Mean Correlation Score per Motif Across Different Parameters\nvolume=(0.07, 0.9), no volume correlation\n(Sorted by sigma=(0.02, 0.4))', fontsize=14)
ax.set_xticks(x_pos)
ax.set_xticklabels(sorted_motifs, ha='right')
ax.grid(True, alpha=0.3, axis='y')
ax.legend()
plt.tight_layout()
#plt.savefig(f'zscore_comparisonwoCorr_{params["vol_param"]}_sigcorr_{params["rho_vol"]}.jpg', dpi=300, bbox_inches='tight')
#plt.savefig(f'zscore_comparisonwoCorr_{params["vol_param"]}_sigcorr_{params["rho_vol"]}.pdf', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# Sort by sigma=(0.02, 0.4) scores in descending order
sorted_indices = np.argsort(data_11['scores'])[::-1]
sorted_motifs = [data_11['motifs'][i] for i in sorted_indices]
sorted_scores_11 = [data_11['scores'][i] for i in sorted_indices]
sorted_vars_11 = [data_11['variances'][i] for i in sorted_indices]

# Create mapping from motif to sorted position
motif_to_pos = {motif: i for i, motif in enumerate(sorted_motifs)}

# Plot comparison of mean correlation scores across different parameters
fig, ax = plt.subplots(figsize=(14, 6))
x_pos = np.arange(len(sorted_motifs))

# Plot sigma=(0.02, 0.4) at the sorted positions
ax.errorbar(x_pos, sorted_scores_11, yerr=sorted_vars_11, fmt='o', color='salmon', ecolor='salmon', capsize=5, label='* volume=(0.07, 0.9)')

# Plot other datasets at matching motif positions
x_11_05 = [motif_to_pos[m] for m in data_11_05['motifs'] if m in motif_to_pos]
scores_11_05 = [data_11_05['scores'][i] for i, m in enumerate(data_11_05['motifs']) if m in motif_to_pos]
vars_11_05 = [data_11_05['variances'][i] for i, m in enumerate(data_11_05['motifs']) if m in motif_to_pos]
ax.errorbar(x_11_05, scores_11_05, yerr=vars_11_05, fmt='o', color='steelblue', ecolor='steelblue', capsize=5, label='volume=(0.07, 0.5)')

x_11_0105 = [motif_to_pos[m] for m in data_11_0105['motifs'] if m in motif_to_pos]
scores_11_0105 = [data_11_0105['scores'][i] for i, m in enumerate(data_11_0105['motifs']) if m in motif_to_pos]
vars_11_0105 = [data_11_0105['variances'][i] for i, m in enumerate(data_11_0105['motifs']) if m in motif_to_pos]
ax.errorbar(x_11_0105, scores_11_0105, yerr=vars_11_0105, fmt='o', color='lightgreen', ecolor='lightgreen', capsize=5, label='volume=(0.1, 0.5)')

ax.set_xlabel('Motif ID', fontsize=12)
ax.set_ylabel('Mean Correlation Score', fontsize=12)
ax.set_title('Mean Correlation Score per Motif Across Different Parameters\nsigma=(1, 1), no volume correlation\n(Sorted by volume=(0.07, 0.9))', fontsize=14)
ax.set_xticks(x_pos)
ax.set_xticklabels(sorted_motifs, ha='right')
ax.grid(True, alpha=0.3, axis='y')
ax.legend()
plt.tight_layout()
#plt.savefig(f'mean_corr_score_comparison_{params["sigma_range"]}_volcorr_{params["rho_vol"]}.jpg', dpi=300, bbox_inches='tight')
#plt.savefig(f'mean_corr_score_comparison_{params["sigma_range"]}_volcorr_{params["rho_vol"]}.pdf', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
#plot mean correlation scores by ranking motifs by their scores independently for each dataset
descending_11 = sorted(zip(data_11['motifs'], data_11['scores'], data_11['variances']), key=lambda x: x[1], reverse=True)
descending_11_05 = sorted(zip(data_11_05['motifs'], data_11_05['scores'], data_11_05['variances']), key=lambda x: x[1], reverse=True)
descending_11_0105 = sorted(zip(data_11_0105['motifs'], data_11_0105['scores'], data_11_0105['variances']), key=lambda x: x[1], reverse=True)

fig, ax = plt.subplots(figsize=(10, 6))
ax.errorbar(range(len([s for m, s, v in descending_11_0105])), [s for m, s, v in descending_11_0105], yerr=[v for m, s, v in descending_11_0105], fmt='o', capsize=5, label='(0.1, 0.5)', color='lightgreen')
ax.errorbar(range(len([s for m, s, v in descending_11_05])), [s for m, s, v in descending_11_05], yerr=[v for m, s, v in descending_11_05], fmt='o', capsize=5, label='(0.07, 0.5)', color='steelblue')
ax.errorbar(range(len([s for m, s, v in descending_11])), [s for m, s, v in descending_11], yerr=[v for m, s, v in descending_11], fmt='o', capsize=5, label='(0.07, 0.9)', color='coral')
ax.set_xticklabels(np.arange(1,21)) 
ax.set_xticks(np.arange(20))     
ax.set_xlim(-0.5, 19.5)  
ax.set_xlabel('Motif Rank', fontsize=15)
ax.set_ylabel('Mean Correlation Score (Z-Score)', fontsize=15)
ax.set_title(f'Mean Correlation Scores per Motif of One Subsample (Independently Ranked)\nσ Range: (1,1)', fontsize=15)
ax.tick_params(axis='y', labelsize=10)
ax.tick_params(axis='x', labelsize=9)
ax.legend(title='Volume Parameters', fontsize=9)
plt.tight_layout()
plt.savefig(f'sig(1,1)_mean_corr_zscores_independent_ranking.jpg', dpi=300)
plt.savefig(f'sig(1,1)_mean_corr_zscores_independent_ranking.pdf', dpi=300)
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(14, 6))

# Get all unique motif IDs present in either dataset
all_motif_ids = set(data_00204_003['motifs']) | set(data_00204_003_07['motifs'])

# Create a mapping of motif IDs to their scores in data_00204_003
motif_to_score_003 = {m: data_00204_003['scores'][i] for i, m in enumerate(data_00204_003['motifs'])}

# Sort all motif IDs by data_00204_003 scores in descending order
sorted_motif_ids = sorted(all_motif_ids, key=lambda m: motif_to_score_003.get(m, 0), reverse=True)

# Create new position mapping based on sorted motif IDs
motif_to_pos_sorted = {motif: i for i, motif in enumerate(sorted_motif_ids)}

# Plot data_00204_003
x_003_new = [motif_to_pos_sorted[m] for m in data_00204_003['motifs']]
scores_003 = [data_00204_003['scores'][i] for i, m in enumerate(data_00204_003['motifs'])]
vars_003 = [data_00204_003['variances'][i] for i, m in enumerate(data_00204_003['motifs'])]
ax.errorbar(x_003_new, scores_003, yerr=vars_003, fmt='o', color='steelblue', ecolor='steelblue', capsize=5, label='ρ=0.0')

# Plot data_00204_003_07
x_003_07_new = [motif_to_pos_sorted[m] for m in data_00204_003_07['motifs']]
scores_003_07 = [data_00204_003_07['scores'][i] for i, m in enumerate(data_00204_003_07['motifs'])]
vars_003_07 = [data_00204_003_07['variances'][i] for i, m in enumerate(data_00204_003_07['motifs'])]
ax.errorbar(x_003_07_new, scores_003_07, yerr=vars_003_07, fmt='o', color='orchid', ecolor='orchid', capsize=5, label='ρ=0.7')

# Set x ticks and labels based on all sorted motif IDs
ax.set_xticks(range(len(sorted_motif_ids)))
ax.set_xticklabels(sorted_motif_ids, ha='right')
ax.set_xlabel('Motif ID', fontsize=12)
ax.set_ylabel('Mean Correlation Score', fontsize=12)
ax.grid(True, alpha=0.3, axis='y')
ax.legend()
plt.title('Mean Correlation Score per Motif Across Different Parameters\nsigma=(0.02, 0.4), vol=(0.03, 0.9)\n(Sorted by volume=(0.03, 0.9) score)', fontsize=14)
plt.tight_layout()
plt.savefig(f'mean_corr_score_comparison_vol_003_{params["rho_vol"]}.jpg', dpi=300, bbox_inches='tight')
plt.savefig(f'mean_corr_score_comparison_vol_003_{params["rho_vol"]}.pdf', dpi=300, bbox_inches='tight')
plt.show()

### Save Retention Results and Volume Medians 

In [ ]:
# Plot median volume of different R for 50 seeds with error bars for standard deviation of median volume across seeds
# add a vertical line for when mean_seqs_retained_pcts is above 50%

n_neurons_keep_values = [10, 20, 30, 50, 75, 100, 150, 200, 350, 500]
seed_range = range(0, 50)  # Seeds from 0 to 49
median_volume_per_seed = {seed: [] for seed in seed_range}
for seed in seed_range:
    for n_keep in n_neurons_keep_values:
        seqs_down, _, _ = downsample_sequences(sequences, spk_times, volume, params['n_neurons'], n_neurons_keep=n_keep, min_length=5, random_state=seed)
        motif_ids = [m for m in seqs_down.keys() if len(seqs_down[m]) > 20]
        median_volumes = [] 
        for m in motif_ids:
            if seqs_down[m]:  # Check if there are sequences for this motif
                median_vol = np.median([volume[neuron, m] for seq in seqs_down[m] for neuron in seq])
                median_volumes.append(median_vol)
        median_volume_per_seed[seed].append(np.mean(median_volumes) if median_volumes else 0)
# Compute mean and std of median volumes across seeds for each n_neurons_keep value
mean_median_volumes = []
std_median_volumes = []
for i in range(len(n_neurons_keep_values)):
    volumes_at_n_keep = [median_volume_per_seed[seed][i] for seed in seed_range]
    mean_median_volumes.append(np.mean(volumes_at_n_keep))
    std_median_volumes.append(np.std(volumes_at_n_keep))


In [ ]:
# Compute median volume of ACTIVE neurons, pooling all 50 subsamples for each n
n_neurons_keep_values = [10, 20, 30, 50, 75, 100, 150, 200, 350, 500]

median_volumes_by_n = {}
median_volumes = {}

for n_keep in n_neurons_keep_values:
    active_volumes_all_seeds = []

    for seed in range(50):
        seqs_down, _, _ = downsample_sequences(
            sequences, spk_times, volume, params['n_neurons'],
            n_neurons_keep=n_keep, min_length=5, random_state=seed
        )
        motif_ids = [m for m in seqs_down if len(seqs_down[m]) > 20]

        # seqs_down: {motif_id: [seq1, seq2, ...]}
        for motif_id in motif_ids:
            for seq in seqs_down[motif_id]:
                if len(seq) == 0:
                    continue
                seq_idx = np.asarray(seq, dtype=int)
                # volume assumed shape: (n_neurons, n_motifs)
                active_volumes_all_seeds.extend(volume[seq_idx, motif_id])

    median_volumes_by_n[n_keep] = active_volumes_all_seeds
    median_volumes[n_keep] = np.median(active_volumes_all_seeds) if active_volumes_all_seeds else np.nan

In [ ]:
median_volumes

In [ ]:
# save the variable (median volumes of ACTIVE NEURONS in sequences across 50 subsamples for each n) as a pickle file
with open(f'whole_mean_median{params["sigma_range"]}_vol_{params["vol_param"]}_volcorr_{params["rho_vol"]}.pkl', 'wb') as f:
    pickle.dump({
        'median_volumes': median_volumes,
        'all_median_volumes_by_n': median_volumes_by_n
    }, f)

In [ ]:
# save the variable 
with open(f'retention_results_{params["sigma_range"]}_vol_{params["vol_param"]}_volcorr_{params["rho_vol"]}.pkl', 'wb') as f:
    pickle.dump({
        'results_per_seed': results_per_seed,
        'mean_seqs_retained_pcts': mean_seqs_retained_pcts,
        'std_seqs_retained_pcts': std_seqs_retained_pcts,
        'mean_motif_counts': mean_motif_counts,
        'std_motif_counts': std_motif_counts,
        'mean_median_volumes': mean_median_volumes,
        'std_median_volumes': std_median_volumes
    }, f)

In [ ]:
# Plot with error bars
plt.figure(figsize=(10, 6))
ax1 = plt.gca()
ax1.errorbar(n_neurons_keep_values, mean_median_volumes, yerr=std_median_volumes, fmt='o-', color='steelblue', ecolor='black', capsize=5)
ax1.set_xlabel('R (Neurons Kept)', fontsize=12)
ax1.set_ylabel('Mean of Median Volumes', fontsize=12)
ax1.set_title(f'Mean of Median Volumes vs R with Error Bars (50 Seeds)\nsigma: {params["sigma_range"]} | volume: {params["vol_param"]}', fontsize=12)
ax1.set_xticks(n_neurons_keep_values)
ax2 = ax1.twinx()
ax2.plot(n_neurons_keep_values, mean_seqs_retained_pcts, marker='o', color='coral', linestyle='--', label='Mean Sequences Retained %')
ax2.set_ylabel('Mean Sequences Retained %', fontsize=12)
ax2.tick_params(axis='y', labelcolor='coral')
ax2.axhline(50, color='red', linewidth=1, label='50% Retention Threshold')
ax2.legend(loc='lower right')    
plt.tight_layout()
plt.savefig(f'mean_median_volume_with_retention_{params["sigma_range"]}_vol_{params["vol_param"]}_volcorr_{params["rho_vol"]}.jpg')
plt.savefig(f'mean_median_volume_with_retention_{params["sigma_range"]}_vol_{params["vol_param"]}_volcorr_{params["rho_vol"]}.jpg')
plt.show()

### Load Retention & Volume Data: (0.02, 0.4) | (0.07, 0.5)

In [ ]:
# load data
with open(f'retention_results_(0.02, 0.4)_vol_(0.07, 0.5)_volcorr_0.0.pkl', 'rb') as f:
    data = pickle.load(f)
with open(f'whole_mean_median(0.02, 0.4)_vol_(0.07, 0.5)_volcorr_0.0.pkl', 'rb') as f:
    data_median_vol = pickle.load(f)

results_05 = data['results_per_seed']
mean_seqs_05 = data['mean_seqs_retained_pcts']
std_seqs_05 = data['std_seqs_retained_pcts']
mean_motif_05 = data['mean_motif_counts']
std_motif_05 = data['std_motif_counts']
mean_median_vol_05 = data['mean_median_volumes']
std_median_vol_05 = data['std_median_volumes']

idx_above50 = np.where(np.array(mean_seqs_05) > 50)[0][0]
n_neurons_keep_values = [10, 20, 30, 50, 75, 100, 150, 200, 350, 500]
R_above50 = n_neurons_keep_values[idx_above50] # R value where sequence retention goes above 50% for vol_param (0.07, 0.9)
First_corr_above50 = mean_seqs_05[idx_above50] # First correlation value where sequence retention goes above 50% for vol_param (0.07, 0.9)
First_motif_above50 = mean_motif_05[idx_above50] # First motif count where sequence retention goes above 50% for vol_param (0.07, 0.9)
First_median_vol_above50 = mean_median_vol_05[idx_above50] # First median volume where sequence retention goes above 50% for vol_param (0.07, 0.9)

print('======= sigma_range: (0.02, 0.4), vol_param: (0.07, 0.5) ========')
print(f"R value where sequence retention goes above 50%: {R_above50}")
print(f"First correlation value where sequence retention goes above 50%: {First_corr_above50:.2f}%")
print(f"First motif count where sequence retention goes above 50%: {First_motif_above50} ({First_motif_above50/20*100:.1f}% of motifs)")
print(f"First median volume where sequence retention goes above 50%: {First_median_vol_above50:.4f}")

### Load Retention & Volume Data: (0.02, 0.4) | (0.07, 0.9) (run std)

In [ ]:
# load data
with open(f'retention_results_(0.02, 0.4)_vol_(0.07, 0.9)_volcorr_0.0.pkl', 'rb') as f:
    data = pickle.load(f)
with open(f'whole_mean_median(0.02, 0.4)_vol_(0.07, 0.9)_volcorr_0.0.pkl', 'rb') as f:
    data_median_vol = pickle.load(f)

results_007 = data['results_per_seed']
mean_seqs_007 = data['mean_seqs_retained_pcts']
#std_seqs_007 = data['std_seqs_retained_pcts']
mean_motif_007 = data['mean_motif_counts']
#std_motif_007 = data['std_motif_counts']
mean_median_vol_007 = data['mean_median_volumes']
std_median_vol_007 = data['std_median_volumes']


idx_above50 = np.where(np.array(mean_seqs_007) > 50)[0][0]
n_neurons_keep_values = [10, 20, 30, 50, 75, 100, 150, 200, 350, 500]
R_above50 = n_neurons_keep_values[idx_above50] # R value where sequence retention goes above 50% for vol_param (0.07, 0.9)
First_corr_above50 = mean_seqs_007[idx_above50] # First correlation value where sequence retention goes above 50% for vol_param (0.07, 0.9)
First_motif_above50 = mean_motif_007[idx_above50] # First motif count where sequence retention goes above 50% for vol_param (0.07, 0.9)
#First_median_vol_above50 = mean_median_vol_007[idx_above50] # First median volume where sequence retention goes above 5０% for vol_param (０.０７, ０.９)

print('======= sigma_range: (0.02, 0.4), vol_param: (0.07, 0.9) ========')
print(f"R value where sequence retention goes above 50%: {R_above50}")
print(f"First correlation value where sequence retention goes above 50%: {First_corr_above50:.2f}%")
print(f"First motif count where sequence retention goes above 50%: {First_motif_above50} ({First_motif_above50/20*100:.1f}% of motifs)")
#print(f"First median volume where sequence retention goes above 50%: {First_median_vol_above50:.4f}")

### Load Retention & Volume Data: (0.02, 0.4) | (0.03, 0.9) (run std)

In [ ]:
# load data
with open(f'retention_results_(0.02, 0.4)_vol_(0.03, 0.9)_volcorr_0.0.pkl', 'rb') as f:
    data = pickle.load(f)
with open(f'whole_mean_median(0.02, 0.4)_vol_(0.03, 0.9)_volcorr_0.0.pkl', 'rb') as f:
    data_median_vol = pickle.load(f)

results_003 = data['results_per_seed']
mean_seqs_003 = data['mean_seqs_retained_pcts']
#std_seqs_003 = data['std_seqs_retained_pcts']
mean_motif_003 = data['mean_motif_counts']
#std_motif_003 = data['std_motif_counts']
mean_median_vol_003 = data['mean_median_volumes']
std_median_vol_003 = data['std_median_volumes']

idx_above50 = np.where(np.array(mean_seqs_003) > 50)[0][0]

R_above50 = n_neurons_keep_values[idx_above50] # R value where sequence retention goes above 50% for vol_param (0.03, 0.9)
First_corr_above50 = mean_seqs_003[idx_above50] # First correlation value where sequence retention goes above 50% for vol_param (0.03, 0.9)
First_motif_above50 = mean_motif_003[idx_above50] # First motif count where sequence retention goes above 50% for vol_param (0.03, 0.9)
First_median_vol_above50 = mean_median_vol_003[idx_above50] # First median volume where sequence retention goes above 50% for vol_param (0.03, 0.9)

print('======= sigma_range: (0.02, 0.4), vol_param: (0.03, 0.9) ========')
print(f"R value where sequence retention goes above 50%: {R_above50}")
print(f"First correlation value where sequence retention goes above 50%: {First_corr_above50:.2f}%")
print(f"First motif count where sequence retention goes above 50%: {First_motif_above50} ({First_motif_above50/20*100:.1f}% of motifs)")
print(f"First median volume where sequence retention goes above 50%: {First_median_vol_above50:.4f}")

### Load Retention & Volume Data: (0.02, 0.4) | (0.03, 0.9) | 0.7 (run std)

In [ ]:
# load data
with open(f'retention_results_(0.02, 0.4)_vol_(0.03, 0.9)_volcorr_0.7.pkl', 'rb') as f:
    data = pickle.load(f)
    

results_003_07 = data['results_per_seed']
mean_seqs_003_07 = data['mean_seqs_retained_pcts']
#std_seqs_003_07 = data['std_seqs_retained_pcts']
mean_motif_003_07 = data['mean_motif_counts']
#std_motif_003_07 = data['std_motif_counts']
mean_median_vol_003_07 = data['mean_median_volumes']
std_median_vol_003_07 = data['std_median_volumes']

idx_above50 = np.where(np.array(mean_seqs_003_07) > 50)[0][0]

R_above50 = n_neurons_keep_values[idx_above50] # R value where sequence retention goes above 50% for vol_param (0.03, 0.9)
First_corr_above50 = mean_seqs_003_07[idx_above50] # First correlation value where sequence retention goes above 50% for vol_param (0.03, 0.9)
First_motif_above50 = mean_motif_003_07[idx_above50] # First motif count where sequence retention goes above 50% for vol_param (0.03, 0.9)
First_median_vol_above50 = mean_median_vol_003_07[idx_above50] # First median volume where sequence retention goes above 50% for vol_param (0.03, 0.9)

print('======= sigma_range: (0.02, 0.4), vol_param: (0.03, 0.9), rho_vol: 0.7 ========')
print(f"R value where sequence retention goes above 50%: {R_above50}")
print(f"First correlation value where sequence retention goes above 50%: {First_corr_above50:.2f}%")
print(f"First motif count where sequence retention goes above 50%: {First_motif_above50} ({First_motif_above50/20*100:.1f}% of motifs)")
print(f"First median volume where sequence retention goes above 50%: {First_median_vol_above50:.4f}")

### Load Retention & Volume Data: (0.02, 0.4) | (0.07, 0.9) | 0.7

In [ ]:
# load data
with open(f'retention_results_(0.02, 0.4)_vol_(0.07, 0.9)_volcorr_0.7.pkl', 'rb') as f:
    data = pickle.load(f)

    
results_007_07 = data['results_per_seed']
mean_seqs_007_07 = data['mean_seqs_retained_pcts']
std_seqs_007_07 = data['std_seqs_retained_pcts']
mean_motif_007_07 = data['mean_motif_counts']
std_motif_007_07 = data['std_motif_counts']
mean_median_vol_007_07 = data['mean_median_volumes']
std_median_vol_007_07 = data['std_median_volumes']

idx_above50 = np.where(np.array(mean_seqs_007_07) > 50)[0][0]

R_above50 = n_neurons_keep_values[idx_above50] # R value where sequence retention goes above 50% for vol_param (0.07, 0.9)
First_corr_above50 = mean_seqs_007_07[idx_above50] # First correlation value where sequence retention goes above 50% for vol_param (0.07, 0.9)
First_motif_above50 = mean_motif_007_07[idx_above50] # First motif count where sequence retention goes above 50% for vol_param (0.07, 0.9)
First_median_vol_above50 = mean_median_vol_007_07[idx_above50] # First median volume where sequence retention goes above 50% for vol_param (0.07, 0.9)

print('======= sigma_range: (0.02, 0.4), vol_param: (0.07, 0.9), rho_vol: 0.7 ========')
print(f"R value where sequence retention goes above 50%: {R_above50}")
print(f"First correlation value where sequence retention goes above 50%: {First_corr_above50:.2f}%")
print(f"First motif count where sequence retention goes above 50%: {First_motif_above50} ({First_motif_above50/20*100:.1f}% of motifs)")
print(f"First median volume where sequence retention goes above 50%: {First_median_vol_above50:.4f}")

### Load Retention & Volume Data: (0.02, 0.04) | (0.07, 0.9)

In [ ]:
# load data
with open(f'retention_results_(0.02, 0.04)_vol_(0.07, 0.9)_volcorr_0.0.pkl', 'rb') as f:
    data = pickle.load(f)

    
results_004 = data['results_per_seed']
mean_seqs_004 = data['mean_seqs_retained_pcts']
std_seqs_004 = data['std_seqs_retained_pcts']
mean_motif_004 = data['mean_motif_counts']
std_motif_004 = data['std_motif_counts']

idx_above50 = np.where(np.array(mean_seqs_004) > 50)[0][0]

R_above50 = n_neurons_keep_values[idx_above50] # R value where sequence retention goes above 50% for vol_param (0.07, 0.9)
First_corr_above50 = mean_seqs_004[idx_above50] # First correlation value where sequence retention goes above 50% for vol_param (0.07, 0.9)
First_motif_above50 = mean_motif_004[idx_above50] # First motif count where sequence retention goes above 50% for vol_param (0.07, 0.9)
#First_median_vol_above50 = mean_median_vol_004[idx_above50] # First median volume where sequence retention goes above 50% for vol_param (0.07, 0.9)

print('======= sigma_range: (0.02, 0.04), vol_param: (0.07, 0.9), rho_vol: 0.0 ========')
print(f"R value where sequence retention goes above 50%: {R_above50}")
print(f"First correlation value where sequence retention goes above 50%: {First_corr_above50:.2f}%")
print(f"First motif count where sequence retention goes above 50%: {First_motif_above50} ({First_motif_above50/20*100:.1f}% of motifs)")
#print(f"First median volume where sequence retention goes above 50%: {First_median_vol_above50:.4f}")

### Load Retention & Volume Data: (0.4, 0.4) | (0.07, 0.9)

In [ ]:
# load data
with open(f'retention_results_(0.4, 0.4)_vol_(0.07, 0.9)_volcorr_0.0.pkl', 'rb') as f:
    data = pickle.load(f)

    
results_04 = data['results_per_seed']
mean_seqs_04 = data['mean_seqs_retained_pcts']
std_seqs_04 = data['std_seqs_retained_pcts']
mean_motif_04 = data['mean_motif_counts']
std_motif_04 = data['std_motif_counts']

idx_above50 = np.where(np.array(mean_seqs_04) > 50)[0][0]

R_above50 = n_neurons_keep_values[idx_above50] # R value where sequence retention goes above 50% for vol_param (0.07, 0.9)
First_corr_above50 = mean_seqs_04[idx_above50] # First correlation value where sequence retention goes above 50% for vol_param (0.07, 0.9)
First_motif_above50 = mean_motif_04[idx_above50] # First motif count where sequence retention goes above 50% for vol_param (0.07, 0.9)
#First_median_vol_above50 = mean_median_vol_04[idx_above50] # First median volume where sequence retention goes above 50% for vol_param (0.07, 0.9)

print('======= sigma_range: (0.4, 0.4), vol_param: (0.07, 0.9), rho_vol: 0.0 ========')
print(f"R value where sequence retention goes above 50%: {R_above50}")
print(f"First correlation value where sequence retention goes above 50%: {First_corr_above50:.2f}%")
print(f"First motif count where sequence retention goes above 50%: {First_motif_above50} ({First_motif_above50/20*100:.1f}% of motifs)")
#print(f"First median volume where sequence retention goes above 50%: {First_median_vol_above50:.4f}")

### Load Retention & Volume Data: (1, 1) | (0.07, 0.9)

In [ ]:
# load data
with open(f'retention_results_(1, 1)_vol_(0.07, 0.9)_volcorr_0.0_sigmacorr_0.0.pkl', 'rb') as f:
    data = pickle.load(f)
    
results_1 = data['results_per_seed']
mean_seqs_1 = data['mean_seqs_retained_pcts']
std_seqs_1 = data['std_seqs_retained_pcts']
mean_motif_1 = data['mean_motif_counts']
std_motif_1 = data['std_motif_counts']

n_neurons_keep_values = [10, 20, 30, 50, 75, 100, 150, 200, 350, 500]
idx_above50 = np.where(np.array(mean_seqs_1) > 50)[0][0]

R_above50 = n_neurons_keep_values[idx_above50] # R value where sequence retention goes above 50% for vol_param (0.07, 0.9)
First_corr_above50 = mean_seqs_1[idx_above50] # First correlation value where sequence retention goes above 50% for vol_param (0.07, 0.9)
First_motif_above50 = mean_motif_1[idx_above50] # First motif count where sequence retention goes above 50% for vol_param (0.07, 0.9)
#First_median_vol_above50 = mean_median_vol_1[idx_above50] # First median volume where sequence retention goes above 50% for vol_param (0.07, 0.9)

print('======= sigma_range: (1, 1), vol_param: (0.07, 0.9), rho_sig: 0.0 ========')
print(f"R value where sequence retention goes above 50%: {R_above50}")
print(f"First correlation value where sequence retention goes above 50%: {First_corr_above50:.2f}%")
print(f"First motif count where sequence retention goes above 50%: {First_motif_above50} ({First_motif_above50/20*100:.1f}% of motifs)")
#print(f"First median volume where sequence retention goes above 50%: {First_median_vol_above50:.4f}")

### Load Retention & Volume Data: (1, 1) | (0.07, 0.9) | sig 0.7

In [ ]:
# load data
with open(f'retention_results_(1, 1)_vol_(0.07, 0.9)_volcorr_0.0_sigmacorr_0.7.pkl', 'rb') as f:
    data = pickle.load(f)
    
results_1_07 = data['results_per_seed']
mean_seqs_1_07 = data['mean_seqs_retained_pcts']
std_seqs_1_07 = data['std_seqs_retained_pcts']
mean_motif_1_07 = data['mean_motif_counts']
std_motif_1_07 = data['std_motif_counts']


idx_above50 = np.where(np.array(mean_seqs_1_07) > 50)[0][0]

R_above50 = n_neurons_keep_values[idx_above50] # R value where sequence retention goes above 50% for vol_param (0.07, 0.9)
First_corr_above50 = mean_seqs_1_07[idx_above50] # First correlation value where sequence retention goes above 50% for vol_param (0.07, 0.9)
First_motif_above50 = mean_motif_1_07[idx_above50] # First motif count where sequence retention goes above 50% for vol_param (0.07, 0.9)
#First_median_vol_above50 = mean_median_vol_1_07[idx_above50] # First median volume where sequence retention goes above 50% for vol_param (0.07, 0.9)

print('======= sigma_range: (1, 1), vol_param: (0.07, 0.9), rho_sig: 0.0 ========')
print(f"R value where sequence retention goes above 50%: {R_above50}")
print(f"First correlation value where sequence retention goes above 50%: {First_corr_above50:.2f}%")
print(f"First motif count where sequence retention goes above 50%: {First_motif_above50} ({First_motif_above50/20*100:.1f}% of motifs)")
#print(f"First median volume where sequence retention goes above 50%: {First_median_vol_above50:.4f}")

In [ ]:
# === FINAL SUMMARY: Median Volume vs R (with 50% retention threshold reference) ===

fig, ax1 = plt.subplots(figsize=(10, 6))

# === LEFT Y-AXIS: Median Volume (Primary) ===
ax1.set_xlabel('R (Neurons Retained)', fontsize=15)
ax1.set_ylabel('Median Volume', fontsize=15, color='black')
ax1.tick_params(axis='y')
ax1.grid(True, alpha=0.3, axis='both')
ax1.set_yscale('log')                       
ax1.tick_params(axis='both', which='both', labelsize=9)

# Plot all three parameter sets with distinct colors and markers
volume_data = [
    ("vol=(0.07,0.9), ρ=0.0", mean_median_vol_007, mean_seqs_007, 'steelblue', 'o'),
    ("vol=(0.03,0.9), ρ=0.0", mean_median_vol_003, mean_seqs_003, 'coral', 's'),
    ("vol=(0.07,0.9), ρ=0.7", mean_median_vol_007_07, std_median_vol_007_07, mean_seqs_007_07, 'purple', '^'),
    ("vol=(0.03,0.9), ρ=0.7", mean_median_vol_003_07, std_median_vol_003_07, mean_seqs_003_07, 'seagreen', 'D'),
    ("vol=(0.07,0.5), ρ=0.0", mean_median_vol_05, mean_seqs_05, 'red', 'v')
]

for label, vol_mean, seqs_retention, color, marker in volume_data:
    ax1.scatter(n_neurons_keep_values, vol_mean, 
                marker=marker, color=color, 
                label=label, linestyle='none', alpha=0.85)

# === RIGHT Y-AXIS: Sequence Retention % (Reference) ===
ax2 = ax1.twinx()
ax2.set_ylabel('Sequence Retention Threshold (%)', fontsize=15, color='black')
ax2.tick_params(axis='y', labelcolor='black')

# Draw horizontal line at 50% retention threshold
ax2.axhline(50, color='black', linestyle='--', linewidth=2.5, alpha=0.7)
ax2.set_ylim([0, 105])
ax2.set_yticks(range(0, 101, 10))

# === FORMATTING ===
ax1.set_xticks(n_neurons_keep_values)
ax1.set_xticklabels(n_neurons_keep_values)

# Combine legends
lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc='lower right', fontsize=11, 
          title='Parameter Sets', title_fontsize=12, framealpha=0.95)

ax1.set_title('Median Volume vs Neuron Threshold (R)\nwith 50% Sequence Retention Reference', 
             fontsize=15, pad=20)

plt.tight_layout()
plt.savefig('ALLfinal_summary_median_volume_with_threshold.jpg', dpi=300, bbox_inches='tight')
plt.savefig('ALLfinal_summary_median_volume_with_threshold.pdf', bbox_inches='tight')
plt.show()

print("✓ Final summary plot (Median Volume) saved!")


In [ ]:
with open(f'whole_mean_median(0.02, 0.4)_vol_(0.07, 0.9)_volcorr_0.7.pkl', 'rb') as f:
    volumes007_07 = pickle.load(f)

median_vol_007_07 =volumes007_07['median_volumes']

with open(f'whole_mean_median(0.02, 0.4)_vol_(0.07, 0.9)_volcorr_0.0.pkl', 'rb') as f:
    volumes007 = pickle.load(f)

median_vol_007 =volumes007['median_volumes']

with open(f'whole_mean_median(0.02, 0.4)_vol_(0.03, 0.9)_volcorr_0.7.pkl', 'rb') as f:
    volumes003_07 = pickle.load(f)

median_vol_003_07 =volumes003_07['median_volumes']

with open(f'whole_mean_median(0.02, 0.4)_vol_(0.03, 0.9)_volcorr_0.0.pkl', 'rb') as f:
    volumes003 = pickle.load(f)

median_vol_003 =volumes003['median_volumes']

with open(f'whole_mean_median(0.02, 0.4)_vol_(0.07, 0.5)_volcorr_0.0.pkl', 'rb') as f:
    volumes05 = pickle.load(f)
    
median_vol_05 =volumes05['median_volumes']

### FIX CANNOT PLOT RIGTH NOW BECAUSE OF NAN ## 

In [ ]:
n_neurons_keep_values = [10, 20, 30, 50, 75, 100, 150, 200, 350, 500]

In [ ]:
# Line of sequence retention and scattered points of median volume for each parameter set, with vertical line at R where retention goes above 50% for each set, x axist sorted by R
fig, ax1 = plt.subplots(figsize=(10, 6))
# === LEFT Y-AXIS: Sequence Retention % (Primary) ===
ax1.set_xlabel('R (Neurons Retained)', fontsize=15)
ax1.tick_params(axis='x', labelsize=9)
ax2 = ax1.twinx()
ax2.set_ylabel('Sequence Retention (%)', fontsize=15)
ax2.set_yticks(range(0, 101, 10))
ax2.tick_params(axis='y', labelsize=10)
ax2.grid(True, alpha=0.3, axis='both', linestyle='--')
# Plot sequence retention lines for each parameter set
retention_data = [
    ("v=(0.07,0.9), ρ=0.0", mean_seqs_007, std_seqs_007, 'steelblue', 'o', R_above50),
    ("v=(0.03,0.9), ρ=0.0", mean_seqs_003, std_seqs_003, 'coral', 's', R_above50),
    ("v=(0.07,0.9), ρ=0.7", mean_seqs_007_07, std_seqs_007_07, 'purple', '^', R_above50),
    ("v=(0.03,0.9), ρ=0.7", mean_seqs_003_07, std_seqs_003_07, 'seagreen', 'D', R_above50),
    ("v=(0.07,0.5), ρ=0.0", mean_seqs_05, std_seqs_05, 'red', 'v', R_above50)
]
for label, retention, std_retention, color, marker, R_thresh in retention_data:
    ax2.plot(n_neurons_keep_values, retention, linewidth=2.5, 
             color=color, label=f"{label} (Retention)", alpha=0.5)
ax2.axhline(50, color='black', linestyle='--', linewidth=2.5, alpha=0.7)
# === RIGHT Y-AXIS: Median Volume (Scatter plot) ===

ax1.set_ylabel('Median of Accumulated Volumes', fontsize=15)
ax1.tick_params(axis='y', labelsize=10)  # Adjust y-axis limit based on max median volume
# Plot median volume scatter points for each parameter set
volume_data = [
    ("v=(0.07,0.9), ρ=0.0", median_vol_007, 'steelblue', 'o'),
    ("v=(0.03,0.9), ρ=0.0", median_vol_003,  'coral', 's'),
    ("v=(0.07,0.9), ρ=0.7", median_vol_007_07, 'purple', '^'),
    ("v=(0.03,0.9), ρ=0.7", median_vol_003_07, 'seagreen', 'D'),
    ("v=(0.07,0.5), ρ=0.0", median_vol_05, 'red', 'v')
]

for label, vol_median, color, marker in volume_data:
    # Handle NaN values
    vol_median = {k: (v if not np.isnan(v) else 0) for k, v in vol_median.items()}
    # Extract values in the same order as n_neurons_keep_values
    vols = [vol_median.get(n, np.nan) for n in n_neurons_keep_values]
    ax1.scatter(n_neurons_keep_values, vols, marker=marker, 
                 color=color, label=f"{label} (Volume)", alpha=0.85)

# === FORMATTING ===
ax1.set_xticks(n_neurons_keep_values) 
ax1.set_xticklabels(n_neurons_keep_values)
ax1.grid(True, alpha=0.3, axis='both')
# Combine legends
lines1, labels1 = ax2.get_legend_handles_labels()
lines2, labels2 = ax1.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc='lower right', fontsize=9, title='Parameter Sets', title_fontsize=11, framealpha=0.95)
ax1.set_title('Sequence Retention & Volume Median vs Subsampled Neuron Count (R)\nwith R Threshold for 50% Retention | σ=(0.02, 0.4)', fontsize=15, pad=20)
plt.tight_layout()
plt.savefig('final_summary_retention_MEDIANvolume_with_thresholds.jpg', dpi=300, bbox_inches='tight')
plt.savefig('final_summary_retention_MEDIANvolume_with_thresholds.pdf', bbox_inches='tight')
plt.show()


In [ ]:
retention_data = [
    ("σ=(0.02,0.4), ρ=0.0", mean_seqs_007, std_seqs_007, 'steelblue', 'o', R_above50),
    ("σ=(0.02,0.04), ρ=0.0", mean_seqs_004, std_seqs_004, 'coral', 's', R_above50),
    ("σ=(0.4,0.4), ρ=0.0", mean_seqs_04, std_seqs_04, 'purple', '^', R_above50),
    ("σ=(1,1), ρ=0.0", mean_seqs_1, std_seqs_1, 'seagreen', 'D', R_above50),
    ("σ=(1,1), ρ=0.7", mean_seqs_1_07, std_seqs_1_07, 'red', 'v', R_above50)
]

plt.figure(figsize=(10, 6))
for label, retention, std_retention, color, marker, R_thresh in retention_data:
    plt.plot(n_neurons_keep_values, retention, linewidth=2.5, color=color, label=f"{label}", marker=marker, alpha=0.5)
plt.axhline(50, color='black', linestyle='--', linewidth=2.5, alpha=0.7)
plt.xlabel('R (Neurons Retained)', fontsize=15)
plt.ylabel('Sequence Retention (%)', fontsize=15)
plt.title('Sequence Retention vs Subsampled Neuron Count (R)\nwith 50% Sequence Retention Reference | v=(0.07, 0.9)', fontsize=15, pad=20)
plt.xticks(n_neurons_keep_values)
plt.grid(True, alpha=0.3, axis='both', linestyle='--')
plt.tick_params(axis='x', labelsize=9)
plt.tick_params(axis='y', labelsize=10)
plt.legend(loc='lower right', fontsize=9, title='Parameter Sets', title_fontsize=11, framealpha=0.95)
plt.tight_layout()
plt.savefig('final_seq_retention_forsigma_with_thresholds.jpg', dpi=300, bbox_inches='tight')
plt.savefig('final_seq_retention_forsigma_with_thresholds.pdf', bbox_inches='tight')
plt.show()

### Exact R value at 50% by Polynomial Fit

In [ ]:
import numpy as np
from scipy.optimize import fsolve

plt.figure(figsize=(10, 6))

retention_data = [
    ("vol=(0.07,0.9), ρ=0.0", mean_seqs_007, std_seqs_007, 'steelblue', 'o', R_above50),
    ("vol=(0.03,0.9), ρ=0.0", mean_seqs_003, std_seqs_003, 'coral', 's', R_above50),
    ("vol=(0.07,0.9), ρ=0.7", mean_seqs_007_07, std_seqs_007_07, 'purple', '^', R_above50),
    ("vol=(0.03,0.9), ρ=0.7", mean_seqs_003_07, std_seqs_003_07, 'seagreen', 'D', R_above50),
    ("vol=(0.07,0.5), ρ=0.0", mean_seqs_05, std_seqs_05, 'red', 'v', R_above50)
]

# Store results for R at 50%
results_at_50 = []

for label, retention, std_retention, color, marker, R_thresh in retention_data:
    # Convert arrays to numpy arrays
    x = np.array(n_neurons_keep_values)
    y = np.array(retention)
    
    # Find last point where y == 0%
    idx_zero = np.where(y == 0.0)[0]
    if len(idx_zero) > 0:
        last_0_idx = idx_zero[-1]
    else:
        # If no 0%, start from first point
        last_0_idx = -1
    
    # Find first point where y >= 99%
    idx_99 = np.where(y >= 99.0)[0]
    if len(idx_99) > 0:
        first_99_idx = idx_99[0]
    else:
        # If 99% is never reached, use the last point
        first_99_idx = len(y) - 1
    
    # Fit polynomial curve from after 0% to 99%
    x_fit = x[last_0_idx + 1:first_99_idx + 1]
    y_fit = y[last_0_idx + 1:first_99_idx + 1]
    coeffs = np.polyfit(x_fit, y_fit, deg=2)
    poly = np.poly1d(coeffs)
    
    # Find R value when y = 50% by solving: poly(x) - 50 = 0
    def equation(x_val):
        return poly(x_val) - 50
    
    # Find root starting from the middle of the fitted region
    r_at_50 = fsolve(equation, x_fit.mean())[0]
    
    # Calculate R² for the fitted portion
    r_squared = np.corrcoef(y_fit, poly(x_fit))[0, 1]**2
    
    results_at_50.append({
        'label': label,
        'coefficients': coeffs,
        'r_squared': r_squared,
        'r_at_50': r_at_50,
        'x_0': x[last_0_idx] if last_0_idx >= 0 else x[0],
        'x_99': x[first_99_idx]
    })
    
    # Plot scatter points
    plt.scatter(x, y, linewidth=2.5, color=color, label=f"{label} R50% ={r_at_50:.2f}", 
               alpha=0.6, s=80, marker=marker, zorder=4)
    
    # Plot any data before first curve point as a horizontal line at 0%
    if last_0_idx >= 0:
        x_before = np.linspace(x[0], x[last_0_idx], 50)
        y_before = np.zeros_like(x_before)
        plt.plot(x_before, y_before, color=color, linestyle='-', linewidth=2.5, alpha=0.9, zorder=3)
    
    # Plot fitted polynomial curve (from after 0% to 99%)
    x_curve = np.linspace(x_fit.min(), x_fit.max(), 150)
    y_curve = poly(x_curve)
    plt.plot(x_curve, y_curve, color=color, linestyle='-', linewidth=2.5, alpha=0.9, zorder=3)
    
    # Plot straight line from 99% to the end of the plot
    x_straight = np.linspace(x[first_99_idx], x.max(), 50)
    y_straight = np.full_like(x_straight, y[first_99_idx])
    plt.plot(x_straight, y_straight, color=color, linestyle='-', linewidth=2.5, alpha=0.9, zorder=3)
    
    # Mark the 50% intersection point
    plt.plot(r_at_50, 50, marker='*', markersize=25, color=color, 
            markeredgecolor='black', markeredgewidth=1.5, zorder=5)

# Add horizontal line at y=50%
plt.axhline(y=50, color='gray', linestyle=':', linewidth=2, alpha=0.5, label='50% threshold')

plt.xlabel('R (Neurons Retained)', fontsize=15)
plt.ylabel('Sequence Retention (%)', fontsize=15)
plt.legend(loc='lower right', fontsize=9, title='Parameter Sets', title_fontsize=11, framealpha=0.95)
plt.grid(True, alpha=0.3, axis='both', linestyle='--')
plt.title('Sequence Retention vs R with 50% Retention Threshold\nPolynomial Fit to Determine R at 50%', fontsize=15, pad=20)
plt.tight_layout()
#plt.savefig('sig_retention_with_polynomial_fit_and_threshold.jpg', dpi=300, bbox_inches='tight')
#plt.savefig('sig_retention_with_polynomial_fit_and_threshold.pdf', bbox_inches='tight')
plt.show()

# Print results
for result in results_at_50:
    print(f"\n{result['label']}")
    print(f"  Zero retention up to R = {result['x_0']:.2f}")
    print(f"  Polynomial curve from R = {result['x_0']:.2f} to R = {result['x_99']:.2f}")
    print(f"  Polynomial coefficients: {result['coefficients']}")
    print(f"  R² (fitted portion): {result['r_squared']:.6f}")
    print(f"  → R value at 50% retention: {result['r_at_50']:.2f}")


In [ ]:
with open(f"50%_retention_R.pkl", 'wb') as f:
    pickle.dump(results_at_50, f)

### write volume to keep sigma constant and vary vol_param 
### write sigma to keep vol_param constant and vary sigma_range

#### Load R at 50%

In [ ]:
with open("sigma_50%_retention_R.pkl", 'rb') as f:
    loaded_results = pickle.load(f)

r_of_each_at_50 = {result['label']: result['r_at_50'] for result in loaded_results}

In [ ]:
with open("volume_50%_retention_R.pkl", 'rb') as f:
    loaded_results = pickle.load(f)

r_of_each_at_50 = {result['label']: result['r_at_50'] for result in loaded_results}

### R at 50% vs Neuron Count - Volume

In [ ]:
### USE THESE VARIABLES ###
with open(f"all_neuron_counts_(0.02, 0.4)_(0.03, 0.9)_0.0.pkl", "rb") as f:
    neurons_003 = pickle.load(f)

with open(f"all_neuron_counts_(0.02, 0.4)_(0.03, 0.9)_0.7.pkl", "rb") as f:
    neurons_003_07 = pickle.load(f)

with open(f"all_neuron_counts_(0.02, 0.4)_(0.07, 0.9)_0.0.pkl", "rb") as f:
    neurons_007 = pickle.load(f)

with open(f"all_neuron_counts_(0.02, 0.4)_(0.07, 0.9)_0.7.pkl", "rb") as f:
    neurons_007_07 = pickle.load(f)

with open(f"all_neuron_counts_(0.02, 0.4)_(0.07, 0.5)_0.0.pkl", "rb") as f:
    neurons_05 = pickle.load(f)

In [ ]:
from scipy.stats import pearsonr

r_007 = r_of_each_at_50["vol=(0.07,0.9), ρ=0.0"]
r_003 = r_of_each_at_50["vol=(0.03,0.9), ρ=0.0"]
r_05 = r_of_each_at_50["vol=(0.07,0.5), ρ=0.0"]
r_003_07 = r_of_each_at_50["vol=(0.03,0.9), ρ=0.7"]
r_007_07 = r_of_each_at_50["vol=(0.07,0.9), ρ=0.7"]

x_R = [r_05, r_007, r_003, r_003_07, r_007_07]
y_neurons = [neurons_05["avg_counts"], neurons_007["avg_counts"], neurons_003["avg_counts"], neurons_003_07["avg_counts"], neurons_007_07["avg_counts"]]
y_error = [neurons_05["sem_counts"], neurons_007["sem_counts"], neurons_003["sem_counts"], neurons_003_07["sem_counts"], neurons_007_07["sem_counts"]]

labels = ["vol=(0.07,0.5), ρ=0.0", "vol=(0.07,0.9), ρ=0.0", "vol=(0.03,0.9), ρ=0.0", "vol=(0.03,0.9), ρ=0.7", "vol=(0.07,0.9), ρ=0.7"]
colors = ['red', 'steelblue', 'coral', 'seagreen', 'purple']

# Compute Pearson correlation between R values and neuron counts
correlation, p_value = pearsonr(x_R, y_neurons)
print(f"Pearson Correlation: {correlation:.4f}")
print(f"P-value: {p_value:.4f}")

plt.figure(figsize=(10, 6))
for i in range(len(x_R)):
    plt.errorbar(x_R[i], y_neurons[i], yerr=y_error[i],  color=colors[i], marker='o', label=labels[i], capsize=5)
z = np.polyfit(x_R, y_neurons, 1)
p = np.poly1d(z)
x_line = np.linspace(min(x_R), max(x_R), 100)
plt.plot(x_line, p(x_line), "k--", alpha=0.5, linewidth=2, label=f'Linear fit (r={correlation:.3f})')

plt.text(0.05, 0.1, f'r = {correlation:.4f}\np-value = {p_value:.4f}', 
         transform=plt.gca().transAxes, fontsize=12, verticalalignment='top',
         bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

plt.legend(loc='upper right', fontsize=10, title='Parameter Sets', title_fontsize=12, framealpha=0.95)
plt.xlabel('R at 50% Sequence Retention', fontsize=15)
plt.ylabel('Average Neuron Participations across 50 Subsamples\nat R=100', fontsize=14)
plt.grid(True, alpha=0.3, axis='both', linestyle='--')
plt.title('Average Neuron Participations vs R where Sequence Retention Crosses 50%\nsigma=(0.02, 0.4) | Pearson correlation applied', fontsize=15, pad=20)
plt.savefig('accumluated_avg_R100_neuron_counts_at_R_50_retentionwSEM.jpg', dpi=300, bbox_inches='tight')
plt.savefig('accumluated_avg_R100_neuron_counts_at_R_50_retentionwSEM.pdf', bbox_inches='tight')
plt.tight_layout()
plt.show()

### R at 50% vs Relative Firing Order of a Neuron - Sigma

In [ ]:
### USE THESE VARIABLES ###
with open(f"neuron_stds_R_100_(0.02, 0.04)_(0.07, 0.9)_0.0.pkl", "rb") as f:
    order_std_004 = pickle.load(f)
with open(f"neuron_stds_R_100_(0.4, 0.4)_(0.07, 0.9)_0.0.pkl", "rb") as f:
    order_std_04 = pickle.load(f)
with open(f"neuron_stds_R_100_(1, 1)_(0.07, 0.9)_0.0.pkl", "rb") as f:
    order_std_1 = pickle.load(f)
with open(f"neuron_stds_R_100_(1, 1)_(0.07, 0.9)_0.7.pkl", "rb") as f:
    order_std_1_07 = pickle.load(f)
with open(f"neuron_stds_R_100_(0.02, 0.4)_(0.07, 0.9)_0.0.pkl", "rb") as f:
    order_std_002 = pickle.load(f)

In [ ]:
total_orders = [order_std_1, order_std_1_07, order_std_04, order_std_002, order_std_004]
final_deviations = []
for order in total_orders:
    neuron_variances = []
    for neuron_id, orders in order['orders_per_neuron'].items():
        if len(orders) >= 2:  # Must appear at least twice
            # Use np.var() instead of np.std()
            neuron_variances.append(np.var(orders))

    # Step 4: Average the variances, THEN take the square root
    if neuron_variances:
        # This is the "average and take the square root" part
        mean_variance = np.mean(neuron_variances)
        final_firing_order_deviation = np.sqrt(mean_variance)
    else:
        final_firing_order_deviation = 0.0
    final_deviations.append(final_firing_order_deviation)

In [ ]:
mean_std_002 = np.mean(order_std_002['neuron_stds'])
mean_std_004 = np.mean(order_std_004['neuron_stds'])
mean_std_04 = np.mean(order_std_04['neuron_stds'])
mean_std_1 = np.mean(order_std_1['neuron_stds'])
mean_std_1_07 = np.mean(order_std_1_07['neuron_stds'])

In [ ]:
final_deviations 

In [ ]:
from scipy.stats import pearsonr

r_1 = r_of_each_at_50["sig=(1, 1), ρ=0.0"]
r_1_07 = r_of_each_at_50["sig=(1, 1), ρ=0.7"]
r_04 = r_of_each_at_50["sig=(0.4,0.4), ρ=0.0"]
r_002 = r_of_each_at_50["sig=(0.02,0.4), ρ=0.0"]
r_004 = r_of_each_at_50["sig=(0.02,0.04), ρ=0.0"]

x_R = [r_1, r_1_07, r_04, r_002, r_004]
y_order = [final_deviations[0], final_deviations[1], final_deviations[2], final_deviations[3], final_deviations[4]]


labels = ["sig=(1, 1), ρ=0.0", "sig=(1, 1), ρ=0.7", "sig=(0.4,0.4), ρ=0.0", "sig=(0.02,0.4), ρ=0.0", "sig=(0.02,0.04), ρ=0.0"]
colors = ['red', 'steelblue', 'coral', 'seagreen', 'purple']

# Compute Pearson correlation between R values and relative firing order
correlation, p_value = pearsonr(x_R, y_order)
print(f"Pearson Correlation: {correlation:.4f}")
print(f"P-value: {p_value:.4f}")

plt.figure(figsize=(10, 6))
for i in range(len(x_R)):
    plt.scatter(x_R[i], y_order[i],  color=colors[i], marker='o', label=labels[i])

plt.text(0.05, 0.4, f'r = {correlation:.4f}\np-value = {p_value:.4f}', 
         transform=plt.gca().transAxes, fontsize=12, verticalalignment='top',
         bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))


plt.legend(loc='lower left', fontsize=10, title='Parameter Sets', title_fontsize=12, framealpha=0.5)
plt.xlabel('R at 50% Sequence Retention', fontsize=15)
plt.ylabel('Pooled Firing Order Deviation', fontsize=13)
plt.grid(True, alpha=0.3, axis='both', linestyle='--')
plt.title('Pooled Firing Order Deviation vs R where Sequence Retention Crosses 50%\n R=100 | vol=(0.07, 0.9) | Pearson correlation applied', fontsize=15, pad=20)
plt.savefig('100R_relativeorder_at_R_50_retention_woError.jpg', dpi=300, bbox_inches='tight')
plt.savefig('100R_relativeorder_at_R_50_retention_woError.pdf', bbox_inches='tight')
plt.tight_layout()
plt.show()

### Mean Median Volume vs R

In [ ]:
plt.figure(figsize=(10, 6))

volume_data = [
    ("vol=(0.07,0.9), ρ=0.0", mean_median_vol_007, std_median_vol_007, 'steelblue', 'o'),
    ("vol=(0.03,0.9), ρ=0.0", mean_median_vol_003, std_median_vol_003, 'coral', 's'),
    ("vol=(0.07,0.9), ρ=0.7", mean_median_vol_007_07, std_median_vol_007_07, 'purple', '^'),
    ("vol=(0.03,0.9), ρ=0.7", mean_median_vol_003_07, std_median_vol_003_07, 'seagreen', 'D'),
    ("vol=(0.07,0.5), ρ=0.0", mean_median_vol_05, std_median_vol_05, 'red', 'v')
]
for label, vol_mean, vol_std, color, marker in volume_data:
    plt.errorbar(n_neurons_keep_values, vol_mean, yerr=vol_std, marker=marker, 
                 color=color, label=f"{label} (Volume)", linestyle='none', alpha=0.85)

# === FORMATTING ===
plt.xticks(n_neurons_keep_values) 
plt.grid(True, alpha=0.3, axis='both')
plt.ylabel('Mean Medians of Volume', fontsize=15)
plt.xlabel('R (Neurons Retained)', fontsize=15)
plt.legend( loc='lower right', fontsize=11, title='Parameter Sets', title_fontsize=12, framealpha=0.95)
plt.title('Mean Median Volume vs Neuron Threshold (R) with Error Bars', fontsize=15, pad=20)
#plt.savefig('final_summary_median_volume_with_error_bars.jpg', dpi=300, bbox_inches='tight')
#plt.savefig('final_summary_median_volume_with_error_bars.pdf', bbox_inches='tight')
plt.tight_layout()
plt.show()

### Std computation for the missing variables

In [ ]:
# === Compute STDs from all results_per_seed dictionaries because it is lackin in the variables===
seed_range = range(0, 50)  # Same seeds as in all computations
n_neurons_keep_values = [10, 20, 30, 50, 75, 100, 150, 200, 350, 500]

# STDs for original results_per_seed
#std_motif_counts = [
#    np.std([results_per_seed[seed]['motif_counts'][i] for seed in seed_range]) 
#    for i in range(len(n_neurons_keep_values))
#]
#std_seqs_retained_pcts = [
#    np.std([results_per_seed[seed]['seqs_retained_pcts'][i] for seed in seed_range]) 
#    for i in range(len(n_neurons_keep_values))
#]

# STDs for results_007
std_motif_007 = [
    np.std([results_007[seed]['motif_counts'][i] for seed in seed_range]) 
    for i in range(len(n_neurons_keep_values))
]
std_seqs_007 = [
    np.std([results_007[seed]['seqs_retained_pcts'][i] for seed in seed_range]) 
    for i in range(len(n_neurons_keep_values))
]

# STDs for results_003
std_motif_003 = [
    np.std([results_003[seed]['motif_counts'][i] for seed in seed_range]) 
    for i in range(len(n_neurons_keep_values))
]
std_seqs_003 = [
    np.std([results_003[seed]['seqs_retained_pcts'][i] for seed in seed_range]) 
    for i in range(len(n_neurons_keep_values))
]

# STDs for results_003_07
std_motif_003_07 = [
    np.std([results_003_07[seed]['motif_counts'][i] for seed in seed_range]) 
    for i in range(len(n_neurons_keep_values))
]
std_seqs_003_07 = [
    np.std([results_003_07[seed]['seqs_retained_pcts'][i] for seed in seed_range]) 
    for i in range(len(n_neurons_keep_values))
]

print("✓ Standard Deviations Computed for All Parameter Sets\n")
print(f"X-axis (R values): {n_neurons_keep_values}\n")
print("=" * 80)
print("vol=(0.07, 0.9), rho=0.0 (Original)")
print(f"Motif STDs:     {std_motif_007}")
print(f"Sequence STDs:  {std_seqs_007}\n")
print("=" * 80)
print("vol=(0.03, 0.9), rho=0.0")
print(f"Motif STDs:     {std_motif_003}")
print(f"Sequence STDs:  {std_seqs_003}\n")
print("=" * 80)
print("vol=(0.03, 0.9), rho=0.7")
print(f"Motif STDs:     {std_motif_003_07}")
print(f"Sequence STDs:  {std_seqs_003_07}")


In [ ]:
# === Create comprehensive summary table with all parameters ===
import pandas as pd

# Create a comprehensive table
data_rows = []

for idx, R_val in enumerate(n_neurons_keep_values):
    row = {'R': R_val}
    
    # vol=(0.07,0.9), ρ=0.0 (Original)
    row['Motifs_007_00'] = f"{mean_motif_007[idx]:.2f}±{std_motif_007[idx]:.2f}"
    row['Seqs_007_00'] = f"{mean_seqs_007[idx]:.2f}±{std_seqs_007[idx]:.2f}"
    row['Volume_007_00'] = f"{mean_median_vol_007[idx]:.2f}±{std_median_vol_007[idx]:.2f}"
    
    # vol=(0.03,0.9), ρ=0.0
    row['Motifs_003_00'] = f"{mean_motif_003[idx]:.2f}±{std_motif_003[idx]:.2f}"
    row['Seqs_003_00'] = f"{mean_seqs_003[idx]:.2f}±{std_seqs_003[idx]:.2f}"
    row['Volume_003_00'] = f"{mean_median_vol_003[idx]:.2f}±{std_median_vol_003[idx]:.2f}"
    
    # vol=(0.03,0.9), ρ=0.7
    row['Motifs_003_07'] = f"{mean_motif_003_07[idx]:.2f}±{std_motif_003_07[idx]:.2f}"
    row['Seqs_003_07'] = f"{mean_seqs_003_07[idx]:.2f}±{std_seqs_003_07[idx]:.2f}"
    row['Volume_003_07'] = f"{mean_median_vol_003_07[idx]:.2f}±{std_median_vol_003_07[idx]:.2f}"
    
    # vol=(0.07,0.9), ρ=0.7 
    row['Motifs_007_07'] = f"{mean_motif_007_07[idx]:.2f}±{std_motif_007_07[idx]:.2f}"
    row['Seqs_007_07'] = f"{mean_seqs_007_07[idx]:.2f}±{std_seqs_007_07[idx]:.2f}"
    row['Volume_007_07'] = f"{mean_median_vol_007_07[idx]:.2f}±{std_median_vol_007_07[idx]:.2f}"

    row['Motifs_05_00'] = f"{mean_motif_05[idx]:.2f}±{std_motif_05[idx]:.2f}"
    row['Seqs_05_00'] = f"{mean_seqs_05[idx]:.2f}±{std_seqs_05[idx]:.2f}"
    row['Volume_05_00'] = f"{mean_median_vol_05[idx]:.2f}±{std_median_vol_05[idx]:.2f}"
    
    data_rows.append(row)

# Create DataFrame
df_summary = pd.DataFrame(data_rows)

# Display the table
print("\n" + "="*150)
print("COMPREHENSIVE SUMMARY TABLE: Motif Counts, Sequence Retention (%), and Median Volume")
print("Format: mean ± std (all values formatted to 2 decimal places)")
print("="*150 + "\n")
print(df_summary.to_string(index=False))
print("\n" + "="*150)

# Save to CSV
csv_filename = 'summary_table_all_parameters.csv'
df_summary.to_csv(csv_filename, index=False)
print(f"\n✓ Table saved to: {csv_filename}")

# Also save a more detailed version with column headers
with open('summary_table_all_parameters_detailed.txt', 'w') as f:
    f.write("COMPREHENSIVE SUMMARY TABLE\n")
    f.write("="*150 + "\n\n")
    f.write("Column Legend:\n")
    f.write("  Motifs_XXX_YY: Mean motif count ± std (vol parameter, rho parameter)\n")
    f.write("  Seqs_XXX_YY:   Mean sequence retention (%) ± std\n")
    f.write("  Volume_XXX_YY: Mean median volume ± std\n\n")
    f.write("Parameters:\n")
    f.write("  007_00: vol=(0.07,0.9), ρ=0.0\n")
    f.write("  003_00: vol=(0.03,0.9), ρ=0.0\n")
    f.write("  003_07: vol=(0.03,0.9), ρ=0.7\n")
    f.write("  007_07: vol=(0.07,0.9), ρ=0.7\n\n")
    f.write("="*150 + "\n\n")
    f.write(df_summary.to_string(index=False))

print(f"✓ Detailed table saved to: summary_table_all_parameters_detailed.txt")


In [ ]:
# Plot the same sequence length per motif plot when R=100 for all parameters together with scattered points of median volume for each motif, x axis sorted by motif id, two y axes one for sequence length and one for median volume, sorted by sequence length, with different colors for each parameter set
R_target = 100
x = np.arange(len(sorted_keys))
width = 0.15

with open(f'median_volume_100_(0.02, 0.4)_(0.07, 0.9)_0.0.pkl', 'rb') as f:
    median_vol_007 = pickle.load(f)

with open(f'median_volume_100_(0.02, 0.4)_(0.07, 0.9)_0.7.pkl', 'rb') as f:
    median_vol_007_07 = pickle.load(f)

with open(f'median_volume_100_(0.02, 0.4)_(0.03, 0.9)_0.0.pkl', 'rb') as f:
    median_vol_003 = pickle.load(f)

with open(f'median_volume_100_(0.02, 0.4)_(0.07, 0.5)_0.0.pkl', 'rb') as f:
    median_vol_05 = pickle.load(f)

### ADD ###
with open(f'median_volume_100_(0.02, 0.4)_vol_(0.03, 0.9)_volcorr_0.7.pkl', 'rb') as f:
    median_vol_003_07 = pickle.load(f)

fig, ax1 = plt.subplots(figsize=(15, 7))
# === LEFT Y-AXIS: Sequence Length per Motif (Primary) ===
ax1.set_xlabel('Motif ID', fontsize=14)
ax1.set_ylabel('Mean Number of Sequences per Motif', fontsize=14)
ax1.tick_params(axis='y')
ax1.grid(True, alpha=0.3, axis='y')

x_007 = x[[i for i, k in enumerate(sorted_keys) if k in seq_counts_007]]
ax1.bar(x_007 - width * 1, [seq_counts_007[k] for k in sorted_keys if k in seq_counts_007], width, color='steelblue', edgecolor='black', alpha=0.7, label='* vol=(0.07, 0.9), ρ=0.0')

x_007_07 = x[[i for i, k in enumerate(sorted_keys) if k in seq_counts_007_07]]
ax1.bar(x_007_07, [seq_counts_007_07[k] for k in sorted_keys if k in seq_counts_007_07], width, color='purple', edgecolor='black', alpha=0.7, label='vol=(0.07, 0.9), ρ=0.7')

x_003 = x[[i for i, k in enumerate(sorted_keys) if k in seq_counts_003]]
ax1.bar(x_003 + width * 1, [seq_counts_003[k] for k in sorted_keys if k in seq_counts_003], width, color='coral', edgecolor='black', alpha=0.7, label='vol=(0.03, 0.9), ρ=0.0')

x_05 = x[[i for i, k in enumerate(sorted_keys) if k in seq_counts_05]]
ax1.bar(x_05 - width * 2, [seq_counts_05[k] for k in sorted_keys if k in seq_counts_05], width, color='red', edgecolor='black', alpha=0.6, label='vol=(0.07, 0.5), ρ=0.0')

x_1 = x[[i for i, k in enumerate(sorted_keys) if k in seq_counts_003_07]]
ax1.bar(x_003_07 + width * 2, [seq_counts_003_07[k] for k in sorted_keys if k in seq_counts_003_07], width, color='seagreen', edgecolor='black', alpha=0.7, label='vol=(0.03, 0.9), ρ=0.7')

ax2 = ax1.twinx()
ax2.set_ylabel('Median Volume', fontsize=14)
ax2.scatter(x_007 - width * 1, [median_vol_007[k] for k in sorted_keys if k in median_vol_007], color='steelblue', marker='o', label='Median Volume (vol=(0.07, 0.9), ρ=0.0)', alpha=0.8)
ax2.scatter(x_007_07, [median_vol_007_07[k] for k in sorted_keys if k in median_vol_007_07], color='purple', marker='s', label='Median Volume (vol=(0.07, 0.9), ρ=0.7)', alpha=0.8)
ax2.scatter(x_003 + width * 1, [median_vol_003[k] for k in sorted_keys if k in median_vol_003], color='coral', marker='D', label='Median Volume (vol=(0.03, 0.9), ρ=0.0)', alpha=0.8)
ax2.scatter(x_05 - width * 2, [median_vol_05[k] for k in sorted_keys if k in median_vol_05], color='red', marker='^', label='Median Volume (vol=(0.07, 0.5), ρ=0.0)', alpha=0.8)
ax2.scatter(x_003_07 + width * 2, [median_vol_003_07[k] for k in sorted_keys if k in median_vol_003_07], color='seagreen', marker='v', label='Median Volume (vol=(0.03, 0.9), ρ=0.7)', alpha=0.8)
ax1.set_xticks(x)
ax1.set_xticklabels(sorted_keys, ha='right')
ax1.set_title(f'Number of Sequences per Motif & Median Volume at R={R_target}\nfor Different Volume Parameters and Correlations\nsigma=(0.02, 0.4)', fontsize=15, pad=20)
ax2.grid(True, alpha=0.3, axis='y', linestyle='--')
fig.legend(loc='upper right', fontsize=8, title='Parameter Sets', title_fontsize=9, framealpha=0.95)
plt.tight_layout()
#plt.savefig(f'sequence_per_motif_with_volume_R_{R_target}.jpg', dpi=300, bbox_inches='tight')
#plt.savefig(f'sequence_per_motif_with_volume_R_{R_target}.pdf', bbox_inches='tight')
plt.show()

In [ ]:
# print the ranges of median volumes for each parameter set at R=100
print(f"Median Volume Ranges at R={R_target}:")
print(f"vol=(0.07, 0.9), ρ=0.0: {min(median_vol_007.values()):.4f} - {max(median_vol_007.values()):.4f}")
print(f"vol=(0.07,  0.9), ρ=0.7: {min(median_vol_007_07.values()):.4f} - {max(median_vol_007_07.values()):.4f}")
print(f"vol=(0.03, 0.9), ρ=0.0: {min(median_vol_003.values()):.4f} - {max(median_vol_003.values()):.4f}")
print(f"vol=(0.07, 0.5), ρ=0.0: {min(median_vol_05.values()):.4f} - {max(median_vol_05.values()):.4f}")
print(f"vol=(0.03, 0.9), ρ=0.7: {min(median_vol_003_07.values()):.4f} - {max(median_vol_003_07.values()):.4f}")

In [ ]:
# Print the volume medians for motifs that have 50% sequence count at R=100 for each parameter set
print(f"Motifs with ≥50% sequence count at R={R_target} and their median volumes:")
for k in sorted_keys:
    if k in seq_counts_007 and seq_counts_007[k] >= 10:  # 10 is 50% of 20 sequences
        print(f"vol=(0.07, 0.9), ρ=0.0 - Motif {k}: Sequence Count = {seq_counts_007[k]}, Median Volume = {median_vol_007[k]:.4f}")
    if k in seq_counts_007_07 and seq_counts_007_07[k] >= 10:
        print(f"vol=(0.07, 0.9), ρ=0.7 - Motif {k}: Sequence Count = {seq_counts_007_07[k]}, Median Volume = {median_vol_007_07[k]:.4f}")
    if k in seq_counts_003 and seq_counts_003[k] >= 10:
        print(f"vol=(0.03, 0.9), ρ=0.0 - Motif {k}: Sequence Count = {seq_counts_003[k]}, Median Volume = {median_vol_003[k]:.4f}")

In [ ]:
# Analyze how mean correlation scores change across different n_neurons_keep values
n_neurons_keep_values = [10, 20, 30, 50, 100, 150, 200, 500]
scores_per_keep = {}  # {n_keep: {motif_id: score}}
vars_per_keep = {}    # {n_keep: {motif_id: var}}

for n_keep in n_neurons_keep_values:
    print(f"Processing n_keep={n_keep}...")
    seqs_down, _, _ = downsample_sequences(sequences, spk_times, volume, params['n_neurons'], n_keep, min_length=5, random_state=1)
    
    # Filter motifs with more than 20 sequences
    motif_ids_keep = [m for m in seqs_down.keys() if len(seqs_down[m]) > 20]
    seqs_down = {m: seqs_down[m] for m in motif_ids_keep}
    
    if not seqs_down:
        print(f"  No motifs with >20 sequences at n_keep={n_keep}, skipping...")
        continue
    
    # Create motif index array
    motif_index = []
    for motif_id in sorted(seqs_down.keys()):
        motif_index.extend([motif_id] * len(seqs_down[motif_id]))
    
    if not motif_index:
        print(f"  No sequences for n_keep={n_keep}")
        continue
    
    motif_index = np.array(motif_index)
    
    # Flatten sequences for correlation computation
    seqs_flat = []
    for motif_id in sorted(seqs_down.keys()):
        seqs_flat.extend(seqs_down[motif_id])
    
    # Compute correlation matrix
    _, _, _, _, _, corrmat = rs.allmot(seqs_flat, nrm)
    
    # Compute within-motif correlation scores with proper mapping
    within_score, within_var = add_within_clust_score(motif_index, corrmat, None)
    
    # Create proper mapping from unique motif IDs to scores/vars
    
    unique_motifs = np.unique(motif_index)
    motif_to_score = {motif: score for motif, score in zip(unique_motifs, within_score)}
    motif_to_var = {motif: var for motif, var in zip(unique_motifs, within_var)}
    
    scores_per_keep[n_keep] = motif_to_score
    vars_per_keep[n_keep] = motif_to_var

In [ ]:
# Find motifs with correlation > 0.4 and their mean volumes
print("=" * 80)
print("MOTIFS WITH CORRELATION > 0.4 AND THEIR MEAN VOLUMES")
print("=" * 80)

high_corr_summary = {}  # {n_keep: [(motif_id, correlation_score, mean_volume), ...]}

for n_keep in n_neurons_keep_values:
    if n_keep not in scores_per_keep:
        print(f"\nn_keep={n_keep}: No data")
        continue
    
    # Find motifs with correlation > 0.4
    high_corr_motifs = []
    for motif_id in sorted(scores_per_keep[n_keep].keys()):
        score = scores_per_keep[n_keep][motif_id]
        if not np.isnan(score) and score > 0.4:
            high_corr_motifs.append((motif_id, score))
    
    if high_corr_motifs:
        # Recompute mean volume for this n_keep
        seqs_down, _, _ = downsample_sequences(sequences, spk_times, volume, params['n_neurons'], n_keep, min_length=5, random_state=1)
        
        # Filter motifs with more than 20 sequences
        motif_ids_keep = [m for m in seqs_down.keys() if len(seqs_down[m]) > 20]
        seqs_down = {m: seqs_down[m] for m in motif_ids_keep}
        
        results = []
        for motif_id, score in high_corr_motifs:
            if motif_id in seqs_down and seqs_down[motif_id]:
                mean_vol = np.mean([volume[neuron, motif_id] for seq in seqs_down[motif_id] for neuron in seq])
                results.append((motif_id, score, mean_vol))
        
        high_corr_summary[n_keep] = results
        print(f"\nn_keep={n_keep} ({len(results)} motifs with corr > 0.4):")
        for motif_id, score, mean_vol in results:
            print(f"  Motif {motif_id:2d}: corr={score:.6f}, mean_vol={mean_vol:.6f}")
    else:
        print(f"\nn_keep={n_keep}: No motifs with correlation > 0.4")
        high_corr_summary[n_keep] = []

print("\n" + "=" * 80)
print("SUMMARY TABLE")
print("=" * 80)
for n_keep in n_neurons_keep_values:
    n_motifs = len(high_corr_summary.get(n_keep, []))
    print(f"n_keep={n_keep:4d}: {n_motifs} motifs with corr > 0.4")


### Neuron Participation

In [ ]:
# Count neuron participation for each seed in each R
# Then average each neuron position's count across 50 seeds within each R

n_neurons_keep_values = [10, 20, 30, 50, 75, 100, 150, 200, 350, 500]
seed_range = range(0, 50)

# Dictionary: {R: [seed1_reindexed_counts, seed2_reindexed_counts, ...]}
neuron_counts_by_R_seed = {n_keep: [] for n_keep in n_neurons_keep_values}

# Step 1: For each seed, count participation in each R
for seed in seed_range:
    for n_keep in n_neurons_keep_values:
        seqs_down, _, _ = downsample_sequences(
                sequences, spk_times, volume, params['n_neurons'], n_neurons_keep=n_keep, min_length=5, random_state=seed)

        # Filter motifs with more than 20 sequences
        motif_ids = [m for m in seqs_down.keys() if len(seqs_down[m]) > 20] 
        seqs_down = {m: seqs_down[m] for m in motif_ids}

        # Count how many sequences each neuron appears in
        seed_neuron_counts = {}
        for motif_id in motif_ids:
            for seq in seqs_down[motif_id]:
                for neuron in seq:
                    if neuron not in seed_neuron_counts:
                        seed_neuron_counts[neuron] = 0
                    seed_neuron_counts[neuron] += 1
        
        # Sort neurons by participation count (descending) and re-index from 1
        sorted_neurons = sorted(seed_neuron_counts.items(), key=lambda x: x[1], reverse=True)
        reindexed_counts = {}
        for new_index, (neuron, count) in enumerate(sorted_neurons, start=1):
            reindexed_counts[new_index] = count
        
        # Store reindexed counts for this seed
        if reindexed_counts:
            neuron_counts_by_R_seed[n_keep].append(reindexed_counts)

# Step 2: For each R, average neuron counts across 50 seeds (neuron_position -> avg_count)
neuron_counts_per_R = {}
neuron_std_per_R = {}
for n_keep in n_neurons_keep_values:
    seed_counts_list = neuron_counts_by_R_seed[n_keep]  # List of 50 dicts
    if seed_counts_list:
        # Find max neuron position across all 50 seeds for this R
        max_neuron_pos = 0
        for counts in seed_counts_list:
            if counts:
                max_neuron_pos = max(max_neuron_pos, max(counts.keys()))
        
        # For each neuron position, average its count across 50 seeds
        averaged_neuron_counts = {}
        std_neuron_counts = {}
        for neuron_pos in range(1, max_neuron_pos + 1):
            counts_at_position = [counts.get(neuron_pos, 0) for counts in seed_counts_list]
            averaged_neuron_counts[neuron_pos] = sum(counts_at_position) / len(seed_counts_list)
            std_neuron_counts[neuron_pos] = np.std(counts_at_position)
        
        # Store averaged counts for this R value
        neuron_counts_per_R[n_keep] = averaged_neuron_counts
        neuron_std_per_R[n_keep] = std_neuron_counts

# Result: neuron_counts_per_R = {10: {1: avg_count, 2: avg_count, ...}, 20: {...}, ...}

In [ ]:
# Take average counts for the whole population of neurons for each R
avg_counts_per_R = {}
std_counts_per_R = {}

for n_keep in n_neurons_keep_values:
    seed_counts_list = neuron_counts_by_R_seed[n_keep]  # List of 50 dicts
    
    if not seed_counts_list:
        avg_counts_per_R[n_keep] = 0
        std_counts_per_R[n_keep] = 0
        continue
        
    # Calculate the average participation for each seed individually
    seed_averages = []
    for seed_dict in seed_counts_list:
        if seed_dict:
            # Average count of all active neurons in THIS specific seed
            seed_avg = sum(seed_dict.values()) / len(seed_dict)
            seed_averages.append(seed_avg)
        else:
            seed_averages.append(0)
            
    avg_counts_per_R[n_keep] = np.mean(seed_averages)
    std_counts_per_R[n_keep] = np.std(seed_averages)

# Final calculation across all R values
final_count_avg = np.mean(list(avg_counts_per_R.values()))
final_count_std = np.std(list(avg_counts_per_R.values()))

In [ ]:
with open(f'averaged_neuron_counts_per_R_{params["sigma_range"]}_{params["vol_param"]}_{params["rho_vol"]}.pkl', 'wb') as f:
    pickle.dump({'neuron_counts_per_R': neuron_counts_per_R, 'neuron_std_per_R': neuron_std_per_R, 'avg_counts_per_R': avg_counts_per_R, 'std_counts_per_R': std_counts_per_R, 'final_count_averages': final_count_avg, 'final_count_stds': final_count_std}, f)

In [ ]:
## Accumulate all neuron counts for each seed. 
# Take the average of all the counts and get one mean value
all_counts = {}
n_keep = 100

for seed in range(0, 50):
    seqs_down, _, _ = downsample_sequences(
        sequences, spk_times, volume, params['n_neurons'], n_neurons_keep=n_keep, min_length=5, random_state=seed)
        
        # Filter motifs with more than 20 sequences
    motif_ids = [m for m in seqs_down.keys() if len(seqs_down[m]) > 20]
    seqs_down = {m: seqs_down[m] for m in motif_ids}

    for motif_id in motif_ids:
        for seq in seqs_down[motif_id]:
            for neuron in seq:
                if neuron not in all_counts:
                    all_counts[neuron] = 0
                all_counts[neuron] += 1  

In [ ]:
avg_counts = sum(all_counts.values()) / len(all_counts)  
sem_counts = np.std(list(all_counts.values())) / np.sqrt(len(all_counts))

In [ ]:
with open(f'all_neuron_counts_{params["sigma_range"]}_{params["vol_param"]}_{params["rho_vol"]}.pkl', 'wb') as f:
    pickle.dump({'all_counts': all_counts, 'avg_counts': avg_counts, 'sem_counts': sem_counts}, f)

In [ ]:
with open(f'all_neuron_counts_(0.02, 0.4)_(0.07, 0.5)_0.0.pkl', 'rb') as f:
    all_counts05 = pickle.load(f)

with open(f'all_neuron_counts_(0.02, 0.4)_(0.07, 0.9)_0.0.pkl', 'rb') as f:
    all_counts007 = pickle.load(f)

with open(f'all_neuron_counts_(0.02, 0.4)_(0.07, 0.9)_0.7.pkl', 'rb') as f:
    all_counts007_07 = pickle.load(f)

with open(f'all_neuron_counts_(0.02, 0.4)_(0.03, 0.9)_0.0.pkl', 'rb') as f:
    all_counts003 = pickle.load(f)

with open(f'all_neuron_counts_(0.02, 0.4)_(0.03, 0.9)_0.7.pkl', 'rb') as f:
    all_counts003_07 = pickle.load(f)

In [ ]:
# Sort by neuron index for plotting
neurons_sorted = sorted(neuron_counts_per_R[100].keys())
counts_sorted = [neuron_counts_per_R[100][n] for n in neurons_sorted]

# Plot
fig, ax = plt.subplots(figsize=(12, 6))
ax.scatter(neurons_sorted, counts_sorted, alpha=0.6, s=30, color='steelblue')
ax.set_xlabel('Neuron Index', fontsize=12)
ax.set_ylabel('Number of Sequences', fontsize=12)
ax.set_title(f'50 Seed | Motifs: {len(motif_ids)} | Active Neurons: {len(neurons_sorted)}', fontsize=11)
ax.grid(True, alpha=0.3)

# Add statistics to the plot
mean_seqs = np.mean(counts_sorted)
max_seqs = np.max(counts_sorted)
ax.axhline(mean_seqs, color='red', linestyle='--', linewidth=1.5, alpha=0.7, label=f'Mean: {mean_seqs:.1f}')
ax.legend(fontsize=10)

plt.suptitle(f'Neuron Participation Across Sequences | sigma: {params["sigma_range"]}, vol: {params["vol_param"]}, rho_vol: {params["rho_vol"]}', 
             fontsize=13, y=0.995)
plt.tight_layout()
plt.savefig(f'neuron_participation_50seed_{params["sigma_range"]}_vol_{params["vol_param"]}_volcorr_{params["rho_vol"]}.jpg', bbox_inches='tight')
plt.savefig(f'neuron_participation_50seed_{params["sigma_range"]}_vol_{params["vol_param"]}_volcorr_{params["rho_vol"]}.pdf', bbox_inches='tight')
plt.show()

In [ ]:
with open(f'averaged_neuron_counts_per_R_(0.02, 0.4)_(0.07, 0.9)_0.0.pkl', 'rb') as f:
    active_007 = pickle.load(f)

with open(f'averaged_neuron_counts_per_R_(0.02, 0.4)_(0.03, 0.9)_0.0.pkl', 'rb') as f:
    active_003 = pickle.load(f)

with open(f'averaged_neuron_counts_per_R_(0.02, 0.4)_(0.03, 0.9)_0.7.pkl', 'rb') as f:
    active_003_07 = pickle.load(f)

with open(f'averaged_neuron_counts_per_R_(0.02, 0.4)_(0.07, 0.5)_0.0.pkl', 'rb') as f:
    active_05 = pickle.load(f)

with open(f'averaged_neuron_counts_per_R_(0.02, 0.4)_(0.07, 0.9)_0.7.pkl', 'rb') as f:
    active_007_07 = pickle.load(f)

In [ ]:
with open(f'all_neuron_counts_(0.02, 0.4)_(0.07, 0.9)_0.0.pkl', 'rb') as f:
    active_007 = pickle.load(f)

with open(f'all_neuron_counts_(0.02, 0.4)_(0.03, 0.9)_0.0.pkl', 'rb') as f:
    active_003 = pickle.load(f)

with open(f'all_neuron_counts_(0.02, 0.4)_(0.03, 0.9)_0.7.pkl', 'rb') as f:
    active_003_07 = pickle.load(f)

with open(f'all_neuron_counts_(0.02, 0.4)_(0.07, 0.5)_0.0.pkl', 'rb') as f:
    active_05 = pickle.load(f)

with open(f'all_neuron_counts_(0.02, 0.4)_(0.07, 0.9)_0.7.pkl', 'rb') as f:
    active_007_07 = pickle.load(f)

In [ ]:
active_003['avg_counts']

In [ ]:
descending_007 = sorted(zip(active_007['neuron_counts_per_R'][100].keys(), active_007['neuron_counts_per_R'][100].values()), key=lambda x: x[1], reverse=True)
descending_003 = sorted(zip(active_003['neuron_counts_per_R'][100].keys(), active_003['neuron_counts_per_R'][100].values()), key=lambda x: x[1], reverse=True)
descending_003_07 = sorted(zip(active_003_07['neuron_counts_per_R'][100].keys(), active_003_07['neuron_counts_per_R'][100].values()), key=lambda x: x[1], reverse=True)
descending_05 = sorted(zip(active_05['neuron_counts_per_R'][100].keys(), active_05['neuron_counts_per_R'][100].values()), key=lambda x: x[1], reverse=True)
descending_007_07 = sorted(zip(active_007_07['neuron_counts_per_R'][100].keys(), active_007_07['neuron_counts_per_R'][100].values()), key=lambda x: x[1], reverse=True)

fig, ax = plt.subplots(figsize=(10, 6))
ax.scatter(range(1, len(descending_007) + 1), [x[1] for x in descending_007], alpha=0.6, s=30, color='steelblue', label='v=(0.07, 0.9), ρ=0.0')
ax.scatter(range(1, len(descending_003) + 1), [x[1] for x in descending_003], alpha=0.6, s=30, color='coral', label='v=(0.03, 0.9), ρ=0.0')
ax.scatter(range(1, len(descending_003_07) + 1), [x[1] for x in descending_003_07], alpha=0.6, s=30, color='seagreen', label='v=(0.03, 0.9), ρ=0.7')
ax.scatter(range(1, len(descending_05) + 1), [x[1] for x in descending_05], alpha=0.6, s=30, color='red', label='v=(0.07, 0.5), ρ=0.0')
ax.scatter(range(1, len(descending_007_07) + 1), [x[1] for x in descending_007_07], alpha=0.6, s=30, color='purple', label='v=(0.07, 0.9), ρ=0.7')

ax.set_xlabel('Neuron Index (ranked within each dataset)', fontsize=15)

ax.set_ylabel('Number of Sequences', fontsize=15)
ax.grid(True, alpha=0.3)
ax.legend(title='Parameter Sets', fontsize=10)
ax.tick_params(axis='y', labelsize=10)
ax.tick_params(axis='x', labelsize=10)
plt.title(f'Average Neuron Participation Across Sequences of 50 Subsamples\nσ=(0.02, 0.4) | R=100', fontsize=15)
plt.tight_layout()
plt.savefig(f'neuron_participation_comparison_50seeds_{params["sigma_range"]}.jpg', bbox_inches='tight')
plt.savefig(f'neuron_participation_comparison_50seeds_{params["sigma_range"]}.pdf', bbox_inches='tight')
plt.show()